# 01 - Fuentes de datos y construcción del dataset documental

## Objetivo del notebook

Este notebook tiene como objetivo construir una base documental fiable para el proyecto de Machine Learning sobre los documentos desclasificados del 23-F.

La pregunta principal que debe responder este notebook es:

> ¿Qué se puede analizar realmente con estos documentos?

Antes de aplicar modelos, necesitamos construir y validar el corpus documental. Para ello, este notebook no empieza directamente con modelos, sino con la creación de una base de datos trazable que permita saber:

- qué documentos existen;
- qué metadatos tiene cada documento;
- qué textos completos pueden utilizarse;
- qué PDFs pueden localizarse o descargarse;
- qué diferencias existen entre las fuentes disponibles;
- qué limitaciones presenta el corpus;
- qué mini casos de uso son viables para la entrega final.

## Productos de datos que se quieren construir

El objetivo técnico del notebook es generar varios productos intermedios. No todos se construyen al principio: algunos son resultado de comparar y validar fuentes.

### 1. Inventario documental de RTVE

Tabla con los metadatos principales de los documentos localizados en el Buscador RTVE 23-F.

Columnas previstas:

- `doc_id`: identificador único del documento.
- `source`: fuente principal del registro.
- `title`: título del documento.
- `pages`: número de páginas.
- `summary`: resumen disponible en RTVE.
- `keywords`: palabras clave disponibles en RTVE.
- `detail_url`: enlace a la página de detalle del documento.
- `pdf_url`: enlace al PDF o documento original, si está disponible.

Este inventario no incluirá el texto completo para evitar crear una tabla demasiado pesada.

### 2. Tabla de textos completos de RTVE

Tabla separada con el texto completo asociado a cada documento.

Columnas previstas:

- `doc_id`: identificador único del documento.
- `text_full`: texto completo del documento.
- `text_length_chars`: longitud del texto en caracteres.
- `text_length_words`: longitud aproximada del texto en palabras.
- `extraction_source`: origen del texto utilizado, por ejemplo, texto HTML de RTVE, OCR o extracción desde PDF.

Además, cuando sea útil, se guardará una copia individual de cada texto en formato `.txt` dentro de `data/raw/texts_rtve/`.

### 3. PDFs de RTVE

Carpeta con los PDFs u originales digitalizados asociados a los documentos de RTVE.

Los archivos se guardarán en:

`data/raw/pdfs_rtve/`

Además, se creará un manifiesto con la relación entre `doc_id`, URL original y ruta local del archivo.

### 4. Inventario oficial de La Moncloa

La Moncloa se utilizará como fuente institucional de contraste. El objetivo no es descargar todos sus PDFs ni duplicar archivos, sino construir un inventario oficial que permita verificar la cobertura de RTVE.

Se creará, si procede:

- `data/interim/inventory_moncloa.csv`

Este inventario se utilizará para comprobar si existen documentos en La Moncloa que no aparezcan en RTVE.

### 5. Comparación RTVE vs La Moncloa

Una vez construidos los inventarios, se compararán ambas fuentes para comprobar:

- qué documentos aparecen en RTVE y en La Moncloa;
- qué documentos aparecen solo en RTVE;
- qué documentos aparecen solo en La Moncloa;
- si hay documentos de La Moncloa que deban incorporarse al corpus final;
- si las diferencias afectan a la cobertura del análisis.

Solo en el caso de detectar documentos presentes en La Moncloa y ausentes en RTVE se valorará descargar esos PDFs concretos y extraer su texto para incorporarlos al corpus.

## Fuentes que se evaluarán

En esta fase se analizarán dos fuentes:

1. **[Buscador RTVE 23-F](https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/)**  
   Fuente principal indicada en el enunciado. Se usará para obtener el corpus base del proyecto: inventario documental, metadatos, textos completos y enlaces finales a PDF.

2. **[Página oficial de La Moncloa sobre la desclasificación de documentos del 23-F](https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx)**  
   Fuente institucional de contraste. Se usará para construir un inventario oficial y verificar si existe algún documento oficial que no aparezca en el corpus obtenido desde RTVE.

## Criterio de trabajo

Cada fuente se evaluará según su utilidad concreta:

- RTVE Buscador: construcción del corpus base, incluyendo metadatos, textos completos y enlaces finales a PDF.
- La Moncloa: validación oficial de cobertura mediante inventario documental.

La prioridad inicial será construir el corpus de trabajo desde RTVE. La Moncloa no se usará para duplicar PDFs, sino para comprobar si existe algún documento oficial ausente en RTVE.

A partir de esa comparación se decidirá:

- qué fuente será la base principal del dataset;
- qué documentos se podrán analizar;
- si hay documentos oficiales que falten en RTVE;
- qué textos son suficientemente útiles para NLP;
- qué mini casos son técnicamente viables.

In [1]:
# ============================================================
# 0. Configuración inicial del notebook
# ============================================================

from pathlib import Path
import os
import sys
import json
import re
import warnings
from collections import Counter
from urllib.parse import urlparse, urljoin

import requests
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0.1. Localización robusta de la raíz del proyecto
# ------------------------------------------------------------

def find_project_root(start_path: Path) -> Path:
    """
    Busca la raíz del proyecto subiendo desde start_path.

    La raíz se identifica por:
    - presencia de carpeta .git, o
    - presencia simultánea de README.md y notebooks/

    Parameters
    ----------
    start_path : Path
        Ruta inicial desde la que se empieza a buscar.

    Returns
    -------
    Path
        Ruta raíz del proyecto.

    Raises
    ------
    FileNotFoundError
        Si no se encuentra una raíz válida.
    """
    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        has_git = (path / ".git").exists()
        has_readme_and_notebooks = (path / "README.md").exists() and (path / "notebooks").exists()

        if has_git or has_readme_and_notebooks:
            return path

    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Ejecuta el notebook desde dentro del repositorio."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# ------------------------------------------------------------
# 0.2. Definición de rutas relativas del proyecto
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

# Datos originales derivados de RTVE
RAW_PDFS_RTVE_DIR = RAW_DIR / "pdfs_rtve"
RAW_TEXTS_RTVE_DIR = RAW_DIR / "texts_rtve"

# Outputs del proyecto
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

SRC_DIR = PROJECT_ROOT / "src"

# Crear carpetas si no existen
for folder in [
    DATA_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    RAW_PDFS_RTVE_DIR,
    RAW_TEXTS_RTVE_DIR,
    OUTPUTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Permitir importar módulos propios desde src/
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# ------------------------------------------------------------
# 0.3. URLs de trabajo
# ------------------------------------------------------------

URL_RTVE_BUSCADOR = "https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/"
URL_MONCLOA = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx"

SOURCE_URLS = {
    "rtve_buscador": URL_RTVE_BUSCADOR,
    "moncloa": URL_MONCLOA,
}

# ------------------------------------------------------------
# 0.4. Cabeceras HTTP comunes
# ------------------------------------------------------------

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# ------------------------------------------------------------
# 0.5. Configuración visual de pandas
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# 0.6. Definición de productos de datos esperados
# ------------------------------------------------------------

# Inventarios documentales
INVENTORY_RTVE_PATH = INTERIM_DIR / "inventory_rtve.csv"
INVENTORY_MONCLOA_PATH = INTERIM_DIR / "inventory_moncloa.csv"

# Textos completos de RTVE
DOCUMENT_TEXTS_RTVE_PATH = INTERIM_DIR / "document_texts_rtve.csv"

# Manifiesto de PDFs RTVE
PDF_MANIFEST_RTVE_PATH = INTERIM_DIR / "pdf_manifest_rtve.csv"

# Comparación y tabla final de documentos validados
SOURCE_COMPARISON_PATH = INTERIM_DIR / "source_comparison.csv"
DOCUMENTS_MASTER_PATH = INTERIM_DIR / "documents_master.csv"
DOCUMENTS_CLEAN_PATH = PROCESSED_DIR / "documents_clean.csv"

# Esquema esperado del inventario documental de RTVE
INVENTORY_RTVE_COLUMNS = [
    "doc_id",
    "source",
    "title",
    "pages",
    "summary",
    "keywords",
    "detail_url",
    "pdf_url",
]

# Esquema esperado del inventario de La Moncloa
INVENTORY_MONCLOA_COLUMNS = [
    "moncloa_id",
    "source",
    "title",
    "pdf_url",
]

# Esquema esperado de la tabla de textos completos
DOCUMENT_TEXT_COLUMNS = [
    "doc_id",
    "text_full",
    "text_length_chars",
    "text_length_words",
    "extraction_source",
]

# Esquema esperado del manifiesto de PDFs RTVE
PDF_MANIFEST_RTVE_COLUMNS = [
    "doc_id",
    "source",
    "title",
    "pdf_url",
    "local_pdf_path",
    "download_ok",
    "file_size_bytes",
]

# Esquema esperado de comparación entre fuentes
SOURCE_COMPARISON_COLUMNS = [
    "doc_id",
    "title",
    "in_rtve",
    "in_moncloa",
    "rtve_pdf_url",
    "moncloa_pdf_url",
    "match_status",
    "notes",
]

# Esquema esperado de la tabla final de documentos validados
DOCUMENTS_MASTER_COLUMNS = [
    "doc_id",
    "title",
    "source_primary",
    "in_rtve",
    "in_moncloa",
    "include_in_corpus",
    "inclusion_reason",
    "pages",
    "summary",
    "keywords",
    "detail_url",
    "pdf_url",
    "text_available",
]

# ------------------------------------------------------------
# 0.7. Comprobación inicial
# ------------------------------------------------------------

print("Configuración inicial completada.")
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Carpeta de datos: {DATA_DIR}")
print(f"Carpeta interim: {INTERIM_DIR}")
print(f"Carpeta processed: {PROCESSED_DIR}")
print(f"Carpeta outputs: {OUTPUTS_DIR}")
print()
print("Productos de datos esperados definidos:")
print(f"- Inventario RTVE: {INVENTORY_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Textos RTVE: {DOCUMENT_TEXTS_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"- PDFs RTVE: {RAW_PDFS_RTVE_DIR.relative_to(PROJECT_ROOT)}")
print(f"- Inventario Moncloa: {INVENTORY_MONCLOA_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Comparación fuentes: {SOURCE_COMPARISON_PATH.relative_to(PROJECT_ROOT)}")
print(f"- Tabla final de documentos validados: {DOCUMENTS_MASTER_PATH.relative_to(PROJECT_ROOT)}")

Configuración inicial completada.
Raíz del proyecto: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML
Carpeta de datos: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data
Carpeta interim: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/interim
Carpeta processed: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/data/processed
Carpeta outputs: /Users/hugo/Desktop/Big Data/Machine Learning/Proyecto ML/ProyectoML/outputs

Productos de datos esperados definidos:
- Inventario RTVE: data/interim/inventory_rtve.csv
- Textos RTVE: data/interim/document_texts_rtve.csv
- PDFs RTVE: data/raw/pdfs_rtve
- Inventario Moncloa: data/interim/inventory_moncloa.csv
- Comparación fuentes: data/interim/source_comparison.csv
- Tabla final de documentos validados: data/interim/documents_master.csv


## 1. Comprobación inicial de fuentes

Antes de construir inventarios, descargar PDFs o extraer textos, se comprueba si las fuentes principales son accesibles desde Python.

Esta comprobación es necesaria porque el proyecto depende de dos niveles de información:

- RTVE como fuente principal para construir el corpus base: inventario, textos completos y enlaces finales a PDF;
- La Moncloa como fuente institucional de contraste para revisar la cobertura documental.

El objetivo de esta sección no es extraer todavía todos los documentos, sino verificar:

- si cada página responde correctamente;
- si devuelve contenido HTML;
- si podemos leer su título;
- si alguna fuente requiere un tratamiento técnico especial;
- si tiene sentido continuar con la extracción automática.

Las fuentes evaluadas son:

1. Buscador RTVE 23-F.
2. Página oficial de La Moncloa.

In [2]:
# ============================================================
# 1. Comprobación inicial de acceso a las fuentes
# ============================================================

def fetch_html(url: str, timeout: int = 20) -> dict:
    """
    Descarga el HTML de una URL y devuelve información básica de diagnóstico.

    Parameters
    ----------
    url : str
        URL que se quiere consultar.
    timeout : int
        Tiempo máximo de espera en segundos.

    Returns
    -------
    dict
        Diccionario con estado de la petición, título HTML y longitud del contenido.
    """
    result = {
        "url": url,
        "domain": urlparse(url).netloc,
        "status_code": None,
        "ok": False,
        "content_type": None,
        "html_length": 0,
        "title": None,
        "error": None,
    }

    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=timeout)
        result["status_code"] = response.status_code
        result["ok"] = response.ok
        result["content_type"] = response.headers.get("Content-Type")
        result["html_length"] = len(response.text) if response.text else 0

        soup = BeautifulSoup(response.text, "html.parser")
        title_tag = soup.find("title")
        result["title"] = title_tag.get_text(strip=True) if title_tag else None

    except Exception as exc:
        result["error"] = str(exc)

    return result


source_diagnostics = []

for source_name, source_url in SOURCE_URLS.items():
    info = fetch_html(source_url)
    info["source_name"] = source_name
    source_diagnostics.append(info)

df_source_diagnostics = pd.DataFrame(source_diagnostics)

cols = [
    "source_name",
    "domain",
    "status_code",
    "ok",
    "content_type",
    "html_length",
    "title",
    "error",
    "url",
]

df_source_diagnostics = df_source_diagnostics[cols]

df_source_diagnostics

,source_name,domain,status_code,ok,content_type,html_length,title,error,url
0,rtve_buscador,www.rtve.es,200,True,text/html; charset=ISO-8859-1,42416,Buscador 23F: explora los documentos desclasificados,None,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/
1,moncloa,www.lamoncloa.gob.es,200,True,text/html; charset=utf-8,149765,La Moncloa. Documentos desclasificados relativos al 23-F | 25/02/2026 [Consejo de Ministros],None,https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx


## 2. Exploración de la estructura HTML del Buscador RTVE

El Buscador RTVE 23-F es la fuente principal para construir el inventario documental y obtener el texto completo de cada documento.

A partir de esta fuente se intentarán extraer dos productos:

1. `inventory_rtve.csv`  
   Tabla ligera con metadatos: `doc_id`, título, páginas, resumen, palabras clave, URL de detalle y posible URL del PDF.

2. `document_texts_rtve.csv` y archivos `.txt`  
   Tabla y ficheros separados con el texto completo de cada documento, relacionados con el inventario mediante `doc_id`.

La separación entre inventario y texto completo es importante porque el inventario debe ser una tabla manejable de metadatos, mientras que el texto completo puede ser largo y se podría utilizar después para NLP, clustering, grafos, búsqueda semántica y análisis de calidad textual.

En esta sección todavía no se construye el inventario completo. Primero se inspecciona la estructura de la página para comprobar:

1. si existen tablas HTML;
2. si existen filas o bloques repetidos de documentos;
3. si los datos aparecen directamente en el HTML inicial;
4. si hay enlaces internos hacia páginas de detalle;
5. si la web usa JavaScript, JSON o algún endpoint/API interno para cargar los resultados.

Esta inspección es necesaria porque, si los documentos no aparecen directamente en el HTML descargado con `requests`, habrá que localizar la fuente dinámica de datos antes de extraer el inventario completo.

In [3]:
# ============================================================
# 2. Exploración de la estructura HTML del Buscador RTVE
# ============================================================

def get_soup(url: str, timeout: int = 20) -> BeautifulSoup:
    """
    Descarga una página HTML y devuelve un objeto BeautifulSoup.

    Parameters
    ----------
    url : str
        URL de la página.
    timeout : int
        Tiempo máximo de espera.

    Returns
    -------
    BeautifulSoup
        HTML parseado.
    """
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=timeout)
    response.raise_for_status()

    # Si el servidor no declara codificación, se usa la detectada por requests.
    if response.encoding is None:
        response.encoding = response.apparent_encoding

    return BeautifulSoup(response.text, "html.parser")


soup_rtve_buscador = get_soup(URL_RTVE_BUSCADOR)

print("HTML descargado correctamente.")
print(f"Título HTML: {soup_rtve_buscador.title.get_text(strip=True) if soup_rtve_buscador.title else None}")
print(f"Número de caracteres del HTML: {len(str(soup_rtve_buscador)):,}")

HTML descargado correctamente.
Título HTML: Buscador 23F: explora los documentos desclasificados
Número de caracteres del HTML: 29,801


In [4]:
# ------------------------------------------------------------
# 2.1. Comprobación de tablas HTML
# ------------------------------------------------------------

tables = soup_rtve_buscador.find_all("table")

print(f"Número de tablas HTML encontradas: {len(tables)}")

for i, table in enumerate(tables[:5], start=1):
    rows = table.find_all("tr")
    print(f"\nTabla {i}: {len(rows)} filas")

    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    if headers:
        print("Cabeceras:", headers)

    first_rows = []
    for row in rows[:3]:
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["td", "th"])]
        first_rows.append(cells)

    print("Primeras filas:")
    for row in first_rows:
        print(row)

Número de tablas HTML encontradas: 0


In [5]:
# ------------------------------------------------------------
# 2.2. Comprobación de si los documentos aparecen en el HTML
# ------------------------------------------------------------

page_text = soup_rtve_buscador.get_text(" ", strip=True)

search_terms = [
    "Vista oral",
    "Consejo Supremo",
    "Resumen",
    "Palabras clave",
    "167 resultados",
    "Texto completo",
]

for term in search_terms:
    found = term.lower() in page_text.lower()
    print(f"'{term}': {found}")

'Vista oral': False
'Consejo Supremo': False
'Resumen': True
'Palabras clave': True
'167 resultados': False
'Texto completo': False


In [6]:
# ------------------------------------------------------------
# 2.3. Inspección de enlaces internos
# ------------------------------------------------------------

links = []

for a in soup_rtve_buscador.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = urljoin(URL_RTVE_BUSCADOR, a["href"])

    if text or "23f" in href.lower():
        links.append({
            "text": text,
            "href": href,
        })

df_links_rtve_buscador = pd.DataFrame(links).drop_duplicates()

print(f"Número de enlaces detectados: {len(df_links_rtve_buscador)}")

df_links_rtve_buscador.head(30)

Número de enlaces detectados: 109


,text,href
0,Saltar al contenido principal,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainContent
1,Saltar al menú de contenido,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#nearNavmenu
2,Saltar al pie de página,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainFooter
3,Volver a navegación principal,https://www.rtve.es/noticias/23f-desclasificados/buscador-23f/#mainNavmenu
4,Pensiones,https://www.rtve.es/noticias/20260430/mapa-pensiones-jubilacion-2026-municipios-espana-datos/17047308.shtml
5,Elecciones de Andalucía,https://www.rtve.es/noticias/20260408/quien-quien-elecciones-andalucia-caras-conocidas-exvicepresidenta-mediran-fuer...
6,Koldo García,https://www.rtve.es/noticias/20260430/koldo-garcia-abalos-tribunal-supremo-caso-mascarillas/17047811.shtml
7,Flotilla,https://www.rtve.es/noticias/20260430/flotilla-rumbo-gaza-interceptada-israel/17047776.shtml
8,Vivienda en España,https://www.rtve.es/noticias/20260427/precio-vivienda-impidio-a-mas-dos-millones-personas-cambiar-casa-ano-pasado/17...
9,Guerra de Irán,https://www.rtve.es/noticias/20260430/guerra-iran-ultima-hora-directo/17047752.shtml


In [7]:
# ------------------------------------------------------------
# 2.4. Inspección de clases CSS frecuentes
# ------------------------------------------------------------

class_counter = Counter()

for tag in soup_rtve_buscador.find_all(True):
    classes = tag.get("class")
    if classes:
        for cls in classes:
            class_counter[cls] += 1

df_classes = (
    pd.DataFrame(class_counter.items(), columns=["class", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

df_classes.head(40)

,class,count
0,ico,65
1,beoff,16
2,blindBox,12
3,tabH1,11
4,wrapper,11
5,brows,9
6,blind,7
7,container,7
8,legend,7
9,datnum,4


## 3. Búsqueda de datos dinámicos en el Buscador RTVE

La exploración anterior muestra que el HTML inicial del Buscador RTVE no contiene directamente los documentos visibles en navegador.

En concreto:

- no se han encontrado tablas HTML;
- no aparece el texto `167 resultados`;
- no aparece `Texto completo`;
- los enlaces detectados pertenecen sobre todo a navegación general de RTVE, no al inventario documental.

Esto sugiere que el buscador carga los documentos mediante JavaScript, JSON embebido o peticiones internas a algún endpoint/API.

El objetivo de esta sección es localizar de dónde obtiene la web los datos necesarios para construir:

1. `inventory_rtve.csv`: inventario documental con título, páginas, resumen, palabras clave y URLs.
2. `document_texts_rtve.csv`: texto completo asociado a cada documento.
3. URLs de detalle o PDFs que permitan relacionar los documentos con su original digitalizado.

Todavía no se extraen todos los documentos. Primero se buscan señales técnicas dentro del HTML y de los scripts cargados por la página.

In [8]:
# ============================================================
# 3. Búsqueda de scripts y recursos internos
# ============================================================

scripts = []

for i, script in enumerate(soup_rtve_buscador.find_all("script"), start=1):
    src = script.get("src")
    script_text = script.get_text(" ", strip=True)

    scripts.append({
        "script_id": i,
        "src": urljoin(URL_RTVE_BUSCADOR, src) if src else None,
        "has_src": src is not None,
        "has_inline_text": bool(script_text),
        "inline_length": len(script_text),
        "contains_23f": "23f" in script_text.lower(),
        "contains_resumen": "resumen" in script_text.lower(),
        "contains_texto": "texto" in script_text.lower(),
        "contains_api": "api" in script_text.lower(),
        "contains_json": "json" in script_text.lower(),
        "inline_preview": script_text[:300] if script_text else None,
    })

df_scripts_rtve = pd.DataFrame(scripts)

print(f"Número de scripts detectados: {len(df_scripts_rtve)}")

df_scripts_rtve

Número de scripts detectados: 5


,script_id,src,has_src,has_inline_text,inline_length,contains_23f,contains_resumen,contains_texto,contains_api,contains_json,inline_preview
0,1,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,True,False,0,False,False,False,False,False,None
1,2,None,False,True,201,True,False,False,False,False,"iFrameResize(\n {\n // opcional: ajusta según necesidad\n checkOrigin: false,\n heightCalculationM..."
2,3,None,False,True,864,False,False,False,False,False,"(function (w, d) {\r\n setTimeout(function () {\r\n if (d.readyState === 'loading') {\r\n d.addEventListene..."
3,4,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,True,False,0,False,False,False,False,False,None
4,5,None,False,True,228,False,False,False,True,False,(function(){ var po=document.createElement('script');po.type='text/javascript';po.async=true;po.src='https://apis.go...


In [9]:
# ------------------------------------------------------------
# 3.1. Búsqueda de términos técnicos en el HTML bruto
# ------------------------------------------------------------

html_rtve = str(soup_rtve_buscador)

technical_terms = [
    "api",
    "json",
    "ajax",
    "fetch",
    "xhr",
    "23f",
    "desclasificados",
    "document",
    "documento",
    "documents",
    "resultados",
    "pagina",
    "page",
    "pages",
    "resumen",
    "keywords",
    "palabras",
    "texto",
    "completo",
    "pdf",
    "original",
]

term_counts = []

for term in technical_terms:
    count = html_rtve.lower().count(term.lower())
    term_counts.append({
        "term": term,
        "count": count,
    })

df_html_term_counts = pd.DataFrame(term_counts).sort_values("count", ascending=False)

df_html_term_counts

,term,count
5,23f,40
6,desclasificados,13
7,document,13
8,documento,9
12,page,8
13,pages,3
17,texto,2
14,resumen,2
1,json,2
0,api,1


In [10]:
# ------------------------------------------------------------
# 3.2. Extracción de URLs candidatas desde el HTML
# ------------------------------------------------------------

url_pattern = r"""https?://[^\s"'<>]+|/[A-Za-z0-9_\-./?=&:%]+"""

raw_urls = re.findall(url_pattern, html_rtve)

candidate_urls = []

for raw_url in raw_urls:
    full_url = urljoin(URL_RTVE_BUSCADOR, raw_url)

    if any(
        term in full_url.lower()
        for term in [
            "23f",
            "desclas",
            "api",
            "json",
            "ajax",
            "document",
            "documento",
            "pdf",
            "buscador",
        ]
    ):
        candidate_urls.append(full_url)

df_candidate_urls = (
    pd.DataFrame({"url": candidate_urls})
    .drop_duplicates()
    .sort_values("url")
    .reset_index(drop=True)
)

print(f"URLs candidatas detectadas: {len(df_candidate_urls)}")

df_candidate_urls.head(100)

URLs candidatas detectadas: 15


,url
0,http://www.rtve.es/buscador/
1,https://23fbuscador.rtve.es/
2,https://apis.google.com/js/plusone.js
3,https://css.rtve.es/css/rtve.2026.noticias/23fdesclasificado-SEC_23FESP/SEC_23FESP.desktp.cab.css
4,https://css.rtve.es/css/rtve.2026.noticias/23fdesclasificado-SEC_23FESP/i/cabecera-23F_logo.png
5,https://www.rtve.es/Especiales/23F
6,https://www.rtve.es/comunes/cabecera/rediseno_2014/portada_2014_flechas_buscador.inc?iso_encoding=true
7,https://www.rtve.es/comunes/cabecera/rediseno_2014/portada_2014_servicio_cabecera.inc?pag=portada/P_NOTICI/SEC_23FES...
8,https://www.rtve.es/comunes/cabecera/rediseno_2014/portada_2014_servicio_cabecera.inc?pag=portada/P_NOTICI/SEC_23FES...
9,https://www.rtve.es/comunes/head/js/head_portada_2014_footer_js.inc?pub_dest=noticias&categoryUid=SEC_23FESP&iso_enc...


In [11]:
# ------------------------------------------------------------
# 3.3. Descarga e inspección de scripts externos
# ------------------------------------------------------------

script_urls = (
    df_scripts_rtve.loc[df_scripts_rtve["has_src"], "src"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

script_diagnostics = []

for script_url in script_urls:
    try:
        response = requests.get(
            script_url,
            headers=REQUEST_HEADERS,
            timeout=20,
        )

        text = response.text or ""
        text_lower = text.lower()

        script_diagnostics.append({
            "script_url": script_url,
            "status_code": response.status_code,
            "ok": response.ok,
            "length": len(text),
            "contains_23f": "23f" in text_lower,
            "contains_api": "api" in text_lower,
            "contains_json": "json" in text_lower,
            "contains_ajax": "ajax" in text_lower,
            "contains_fetch": "fetch" in text_lower,
            "contains_documento": "documento" in text_lower,
            "contains_resumen": "resumen" in text_lower,
            "contains_texto": "texto" in text_lower,
            "contains_pdf": "pdf" in text_lower,
            "preview": text[:300],
            "error": None,
        })

    except Exception as exc:
        script_diagnostics.append({
            "script_url": script_url,
            "status_code": None,
            "ok": False,
            "length": 0,
            "contains_23f": False,
            "contains_api": False,
            "contains_json": False,
            "contains_ajax": False,
            "contains_fetch": False,
            "contains_documento": False,
            "contains_resumen": False,
            "contains_texto": False,
            "contains_pdf": False,
            "preview": None,
            "error": str(exc),
        })

df_script_diagnostics = pd.DataFrame(script_diagnostics)

df_script_diagnostics = df_script_diagnostics.sort_values(
    [
        "contains_23f",
        "contains_api",
        "contains_json",
        "contains_ajax",
        "contains_fetch",
        "contains_documento",
        "contains_texto",
        "length",
    ],
    ascending=False,
).reset_index(drop=True)

df_script_diagnostics

,script_url,status_code,ok,length,contains_23f,contains_api,contains_json,contains_ajax,contains_fetch,contains_documento,contains_resumen,contains_texto,contains_pdf,preview,error
0,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,200,True,13560,False,False,True,False,False,False,False,False,False,/*! iFrame Resizer (iframeSizer.min.js ) - v4.2.11 - 2020-09-09\n * Desc: Force cross domain iframes to size to con...,None
1,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,200,True,16954,False,False,False,False,True,False,False,False,False,"/** vim: et:ts=4:sw=4:sts=4\n * @license RequireJS 2.1.15 Copyright (c) 2010-2014, The Dojo Foundation All Rights Re...",None


In [12]:
# ------------------------------------------------------------
# 3.4. Extracción de URLs candidatas dentro de scripts externos
# ------------------------------------------------------------

script_candidate_urls = []

for script_url in script_urls:
    try:
        response = requests.get(
            script_url,
            headers=REQUEST_HEADERS,
            timeout=20,
        )
        text = response.text or ""

        urls_in_script = re.findall(url_pattern, text)

        for raw_url in urls_in_script:
            full_url = urljoin(script_url, raw_url)

            if any(
                term in full_url.lower()
                for term in [
                    "23f",
                    "desclas",
                    "api",
                    "json",
                    "ajax",
                    "document",
                    "documento",
                    "pdf",
                    "buscador",
                ]
            ):
                script_candidate_urls.append({
                    "source_script": script_url,
                    "candidate_url": full_url,
                })

    except Exception:
        continue

df_script_candidate_urls = pd.DataFrame(
    script_candidate_urls,
    columns=["source_script", "candidate_url"]
)

if df_script_candidate_urls.empty:
    print("No se han detectado URLs candidatas dentro de los scripts externos.")
else:
    df_script_candidate_urls = (
        df_script_candidate_urls
        .drop_duplicates()
        .sort_values(["source_script", "candidate_url"])
        .reset_index(drop=True)
    )

    print(f"URLs candidatas detectadas dentro de scripts: {len(df_script_candidate_urls)}")

df_script_candidate_urls.head(100)

No se han detectado URLs candidatas dentro de los scripts externos.


,source_script,candidate_url


In [13]:
# ------------------------------------------------------------
# 3.5. Fragmentos relevantes dentro de scripts
# ------------------------------------------------------------

keywords_to_inspect = [
    "23f",
    "desclasificados",
    "documento",
    "resumen",
    "texto",
    "pdf",
    "api",
    "json",
    "ajax",
    "fetch",
]

script_snippets = []

def extract_context(text: str, keyword: str, window: int = 180) -> list:
    """
    Extrae fragmentos de contexto alrededor de una palabra clave.
    """
    text_lower = text.lower()
    keyword_lower = keyword.lower()

    snippets = []
    start = 0

    while True:
        idx = text_lower.find(keyword_lower, start)
        if idx == -1:
            break

        left = max(idx - window, 0)
        right = min(idx + len(keyword) + window, len(text))

        snippets.append(text[left:right].replace("\n", " ").replace("\t", " "))
        start = idx + len(keyword)

        if len(snippets) >= 5:
            break

    return snippets


for script_url in script_urls:
    try:
        response = requests.get(
            script_url,
            headers=REQUEST_HEADERS,
            timeout=20,
        )
        text = response.text or ""

        for keyword in keywords_to_inspect:
            snippets = extract_context(text, keyword)

            for snippet in snippets:
                script_snippets.append({
                    "script_url": script_url,
                    "keyword": keyword,
                    "snippet": snippet,
                })

    except Exception:
        continue

df_script_snippets = pd.DataFrame(script_snippets)

print(f"Fragmentos relevantes encontrados: {len(df_script_snippets)}")

df_script_snippets.head(50)

Fragmentos relevantes encontrados: 9


,script_url,keyword,snippet
0,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,json,"etTimeout(function(){k[i]=null,e()},n))}(function(){B(""Send Page Info"",""pageInfo:""+function(){var e=document.body.ge..."
1,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,json,"tRun=!1),b.type){case""close"":C(b.iframe);break;case""message"":!function(e){R(y,""onMessage passed: {iframe: ""+b.iframe..."
2,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,json,"){R(y,""onMessage passed: {iframe: ""+b.iframe.id+"", message: ""+e+""}""),d(""onMessage"",{iframe:b.iframe,message:JSON.par..."
3,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.min.js,json,"ind(null,""Window resize"",""resize"",M[c].iframe),moveToAnchor:function(e){B(""Move to anchor"",""moveToAnchor:""+e,M[c].if..."
4,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,fetch,"p=!0;if(!x){if(x=!0,eachProp(k,function(e){var i=e.map,u=i.id;if(e.enabled&&(i.isDefine||s.push(e),!e.error))if(!e.i..."
5,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,fetch,"his.ignore=r.ignore,r.enabled||this.enabled?this.enable():this.check())},defineDep:function(e,t){this.depMatched[e]|..."
6,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,fetch,"led||this.enabled?this.enable():this.check())},defineDep:function(e,t){this.depMatched[e]||(this.depMatched[e]=!0,th..."
7,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,fetch,"led?this.enable():this.check())},defineDep:function(e,t){this.depMatched[e]||(this.depMatched[e]=!0,this.depCount-=1..."
8,https://js2.rtve.es/pages/pf-portada/2.22.1/js/vendor/require.js,fetch,".depMaps)),d(i),this.defined=!0}this.defining=!1,this.defined&&!this.defineEmitted&&(this.defineEmitted=!0,this.emit..."


### 3.6. Conclusión provisional de la búsqueda dinámica

La inspección de la página principal del Buscador RTVE indica que los documentos no están disponibles directamente en el HTML inicial.

Los scripts externos cargados por la página son genéricos y no contienen el inventario documental ni el texto completo. Sin embargo, dentro de las URLs candidatas aparece:

`https://23fbuscador.rtve.es/`

Esto sugiere que la página principal de RTVE actúa como contenedor y que la aplicación real del buscador se carga desde ese dominio.

Por tanto, el siguiente paso será inspeccionar directamente `https://23fbuscador.rtve.es/` para comprobar si ahí aparecen:

- los 167 documentos;
- los metadatos del inventario;
- los textos completos;
- las URLs de detalle;
- posibles endpoints/API internos;
- la lógica de paginación, incluyendo la opción de mostrar 200 resultados por página.

## 4. Exploración directa de la aplicación `23fbuscador.rtve.es`

La sección anterior ha detectado que la página principal de RTVE no contiene directamente el inventario documental, sino que parece actuar como contenedor de una aplicación externa:

`https://23fbuscador.rtve.es/`

En esta sección se inspecciona directamente esa aplicación para comprobar si ahí aparecen los datos necesarios para construir:

- `inventory_rtve.csv`;
- `document_texts_rtve.csv`;
- URLs de detalle;
- URLs de PDFs u originales digitalizados;
- endpoints/API internos;
- parámetros de paginación, incluyendo la opción de mostrar 200 resultados por página.

El objetivo sigue siendo exploratorio: antes de extraer los 167 documentos, necesitamos entender cómo carga la aplicación los datos.

In [14]:
# ============================================================
# 4. Exploración directa de la aplicación 23fbuscador.rtve.es
# ============================================================

URL_RTVE_APP = "https://23fbuscador.rtve.es/"

def fetch_response(url: str, timeout: int = 20) -> requests.Response:
    """
    Descarga una URL y devuelve el objeto Response.

    Esta función se usa cuando necesitamos inspeccionar no solo el HTML,
    sino también cabeceras, codificación y texto bruto.
    """
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=timeout)
    response.raise_for_status()

    if response.encoding is None:
        response.encoding = response.apparent_encoding

    return response


response_rtve_app = fetch_response(URL_RTVE_APP)
html_rtve_app = response_rtve_app.text
soup_rtve_app = BeautifulSoup(html_rtve_app, "html.parser")

print("Aplicación RTVE descargada correctamente.")
print(f"URL: {URL_RTVE_APP}")
print(f"Status code: {response_rtve_app.status_code}")
print(f"Content-Type: {response_rtve_app.headers.get('Content-Type')}")
print(f"Codificación: {response_rtve_app.encoding}")
print(f"Número de caracteres del HTML: {len(html_rtve_app):,}")
print(f"Título HTML: {soup_rtve_app.title.get_text(strip=True) if soup_rtve_app.title else None}")

Aplicación RTVE descargada correctamente.
URL: https://23fbuscador.rtve.es/
Status code: 200
Content-Type: text/html; charset=utf-8
Codificación: utf-8
Número de caracteres del HTML: 39,685
Título HTML: 23F Documentos Desclasificados


In [15]:
# ------------------------------------------------------------
# 4.1. Comprobación de contenido visible en la aplicación
# ------------------------------------------------------------

app_text = soup_rtve_app.get_text(" ", strip=True)

search_terms_app = [
    "167 resultados",
    "Texto completo",
    "Resumen",
    "Palabras clave",
    "Vista oral",
    "Consejo Supremo",
    "200",
    "página",
    "documentos",
    "desclasificados",
]

for term in search_terms_app:
    found = term.lower() in app_text.lower()
    print(f"'{term}': {found}")

print()
print("Primeros 1.000 caracteres de texto visible:")
print(app_text[:1000])

'167 resultados': True
'Texto completo': False
'Resumen': True
'Palabras clave': True
'Vista oral': True
'Consejo Supremo': True
'200': True
'página': True
'documentos': True
'desclasificados': True

Primeros 1.000 caracteres de texto visible:
23F Documentos Desclasificados 23F Documentos Desclasificados Visor OCR + transcripciones Listado de ficheros Anterior Página 1 de 7 Siguiente × Buscar 167 resultados Búsqueda avanzada 25 / página 50 / página 100 / página 200 / página Que contenga todos estos términos (AND) Que contenga cualquiera de estos términos (OR) Que no contenga estos términos (NOT) Título Páginas KB Resumen Palabras clave Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982). 3 - El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y certificaciones oficiales, aunque plagado de controversias por la interpretación y selección de tes

In [16]:
# ------------------------------------------------------------
# 4.2. Inspección de recursos cargados por la aplicación
# ------------------------------------------------------------

app_resources = []

# Scripts
for tag in soup_rtve_app.find_all("script"):
    src = tag.get("src")
    inline_text = tag.get_text(" ", strip=True)

    app_resources.append({
        "tag": "script",
        "url": urljoin(URL_RTVE_APP, src) if src else None,
        "has_url": src is not None,
        "inline_length": len(inline_text),
        "inline_preview": inline_text[:250] if inline_text else None,
    })

# CSS / links
for tag in soup_rtve_app.find_all("link"):
    href = tag.get("href")
    rel = " ".join(tag.get("rel", [])) if tag.get("rel") else None

    app_resources.append({
        "tag": "link",
        "url": urljoin(URL_RTVE_APP, href) if href else None,
        "has_url": href is not None,
        "rel": rel,
        "inline_length": 0,
        "inline_preview": None,
    })

df_app_resources = pd.DataFrame(app_resources)

print(f"Recursos detectados en la app: {len(df_app_resources)}")

df_app_resources

Recursos detectados en la app: 4


,tag,url,has_url,inline_length,inline_preview,rel
0,script,https://23fbuscador.rtve.es/static/rtve_mushroom.js,True,0,None,NaN
1,script,None,False,5707,"(function () {\n const form = document.querySelector(""form.filters"");\n const toggle = document.getEle...",NaN
2,script,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,True,0,None,NaN
3,link,https://23fbuscador.rtve.es/static/buscador-23F.css,True,0,None,stylesheet


In [17]:
# ------------------------------------------------------------
# 4.3. Descarga e inspección de scripts de la aplicación
# ------------------------------------------------------------

app_script_urls = (
    df_app_resources
    .query("tag == 'script' and has_url == True")["url"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

app_script_diagnostics = []

for script_url in app_script_urls:
    try:
        response = fetch_response(script_url)
        text = response.text or ""
        text_lower = text.lower()

        app_script_diagnostics.append({
            "script_url": script_url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("Content-Type"),
            "length": len(text),
            "contains_167": "167" in text_lower,
            "contains_200": "200" in text_lower,
            "contains_api": "api" in text_lower,
            "contains_json": "json" in text_lower,
            "contains_fetch": "fetch" in text_lower,
            "contains_axios": "axios" in text_lower,
            "contains_documento": "documento" in text_lower,
            "contains_documentos": "documentos" in text_lower,
            "contains_resumen": "resumen" in text_lower,
            "contains_palabras": "palabras" in text_lower,
            "contains_texto": "texto" in text_lower,
            "contains_pdf": "pdf" in text_lower,
            "preview": text[:300],
            "error": None,
        })

    except Exception as exc:
        app_script_diagnostics.append({
            "script_url": script_url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "length": 0,
            "contains_167": False,
            "contains_200": False,
            "contains_api": False,
            "contains_json": False,
            "contains_fetch": False,
            "contains_axios": False,
            "contains_documento": False,
            "contains_documentos": False,
            "contains_resumen": False,
            "contains_palabras": False,
            "contains_texto": False,
            "contains_pdf": False,
            "preview": None,
            "error": str(exc),
        })

df_app_script_diagnostics = pd.DataFrame(app_script_diagnostics)

df_app_script_diagnostics.sort_values(
    [
        "contains_documentos",
        "contains_documento",
        "contains_resumen",
        "contains_texto",
        "contains_api",
        "contains_fetch",
        "length",
    ],
    ascending=False,
).reset_index(drop=True)

,script_url,status_code,ok,content_type,length,contains_167,contains_200,contains_api,contains_json,contains_fetch,contains_axios,contains_documento,contains_documentos,contains_resumen,contains_palabras,contains_texto,contains_pdf,preview,error
0,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,200,True,application/javascript,34420,False,False,True,True,False,False,False,False,False,False,False,False,/*\n * File: iframeResizer.contentWindow.js\n * Desc: Include this file in any page being loaded into an iframe\n * ...,None
1,https://23fbuscador.rtve.es/static/rtve_mushroom.js,200,True,application/javascript; charset=utf-8,1035,False,False,False,False,False,False,False,False,False,False,False,False,"var initGoogleTagManager = (w, d, s, l, i) => {\n\tw[l] = w[l] || [];\n\tw[l].push({ ""gtm.start"": new Date().getTime...",None


In [18]:
# ------------------------------------------------------------
# 4.4. Extracción de URLs candidatas desde la app y sus scripts
# ------------------------------------------------------------

def extract_candidate_urls_from_text(text: str, base_url: str) -> list:
    """
    Extrae URLs absolutas y relativas de un texto y filtra candidatas
    relacionadas con datos, documentos o endpoints.
    """
    raw_urls = re.findall(url_pattern, text)

    candidates = []

    for raw_url in raw_urls:
        full_url = urljoin(base_url, raw_url)

        if any(
            term in full_url.lower()
            for term in [
                "23f",
                "api",
                "json",
                "document",
                "documento",
                "documentos",
                "desclas",
                "pdf",
                "search",
                "buscar",
                "buscador",
                "page",
                "limit",
                "results",
            ]
        ):
            candidates.append(full_url)

    return candidates


app_candidate_urls = []

# Candidatas desde HTML principal
for candidate in extract_candidate_urls_from_text(html_rtve_app, URL_RTVE_APP):
    app_candidate_urls.append({
        "source": "app_html",
        "source_url": URL_RTVE_APP,
        "candidate_url": candidate,
    })

# Candidatas desde scripts
for script_url in app_script_urls:
    try:
        response = fetch_response(script_url)
        text = response.text or ""

        for candidate in extract_candidate_urls_from_text(text, script_url):
            app_candidate_urls.append({
                "source": "app_script",
                "source_url": script_url,
                "candidate_url": candidate,
            })

    except Exception:
        continue

df_app_candidate_urls = pd.DataFrame(
    app_candidate_urls,
    columns=["source", "source_url", "candidate_url"]
)

if df_app_candidate_urls.empty:
    print("No se han detectado URLs candidatas en la app ni en sus scripts.")
else:
    df_app_candidate_urls = (
        df_app_candidate_urls
        .drop_duplicates()
        .sort_values(["source", "candidate_url"])
        .reset_index(drop=True)
    )

    print(f"URLs candidatas detectadas: {len(df_app_candidate_urls)}")

df_app_candidate_urls.head(150)

URLs candidatas detectadas: 88


,source,source_url,candidate_url
0,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/.test
1,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/11-3-82
2,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/3.923/09-03-82
3,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/4.749/23-03-82
4,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/5.169/30-03-82
...,...,...,...
83,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/th
84,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/thead
85,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/title
86,app_html,https://23fbuscador.rtve.es/,https://23fbuscador.rtve.es/tr


In [19]:
# ------------------------------------------------------------
# 4.5. Fragmentos relevantes dentro de los scripts de la app
# ------------------------------------------------------------

app_keywords_to_inspect = [
    "167",
    "200",
    "api",
    "json",
    "fetch",
    "axios",
    "documento",
    "documentos",
    "resumen",
    "palabras",
    "texto",
    "pdf",
    "page",
    "limit",
    "offset",
    "size",
    "search",
    "buscador",
]

app_script_snippets = []

for script_url in app_script_urls:
    try:
        response = fetch_response(script_url)
        text = response.text or ""

        for keyword in app_keywords_to_inspect:
            snippets = extract_context(text, keyword, window=220)

            for snippet in snippets:
                app_script_snippets.append({
                    "script_url": script_url,
                    "keyword": keyword,
                    "snippet": snippet,
                })

    except Exception:
        continue

df_app_script_snippets = pd.DataFrame(app_script_snippets)

print(f"Fragmentos relevantes encontrados en scripts de la app: {len(df_app_script_snippets)}")

df_app_script_snippets.head(80)

Fragmentos relevantes encontrados en scripts de la app: 23


,script_url,keyword,snippet
0,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,api,"r(el, evt, func, options) { el.addEventListener(evt, func, passiveSupported ? options || {} : false) } func..."
1,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,api,ames && Array.prototype.map) { options.eventName = options.eventNames[0] options.eventNames.map(listener...
2,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,api,"dTimer + 'ms') } } // Idea from https://github.com/guardian/iframe-messenger function getMaxElement(side,..."
3,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,json,"""'. The old method will be removed in the next major version."" ) } } function readDataFromPage() { ..."
4,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,json,autoResize = true startEventListeners() } else if (false === resize && true === autoResize) { ...
5,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,json,"t: function scrollToF(x, y) { sendMsg(y, x, 'scrollToOffset') // X&Y reversed at sendMsg uses height/width ..."
6,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,json,"this.moveToAnchor() }, // Backward compatability pageInfo: function pageInfoFromParent() { var ..."
7,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,json,"}, message: function messageFromParent() { var msgBody = getData() log('onMessage called fro..."
8,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,page,/* * File: iframeResizer.contentWindow.js * Desc: Include this file in any page being loaded into an iframe * ...
9,https://www.rtve.es/js/iframe-resizer/js/iframeResizer.contentWindow.js,page,/* * File: iframeResizer.contentWindow.js * Desc: Include this file in any page being loaded into an iframe * ...


In [20]:
# ------------------------------------------------------------
# 4.6. Búsqueda de JSON embebido en el HTML de la app
# ------------------------------------------------------------

json_like_blocks = []

# Scripts inline
for i, script in enumerate(soup_rtve_app.find_all("script"), start=1):
    script_text = script.get_text("\n", strip=True)

    if not script_text:
        continue

    looks_like_json = (
        "{" in script_text
        and "}" in script_text
        and any(term in script_text.lower() for term in ["documento", "resumen", "texto", "23f", "pdf"])
    )

    if looks_like_json:
        json_like_blocks.append({
            "script_id": i,
            "length": len(script_text),
            "preview": script_text[:1000],
        })

df_json_like_blocks = pd.DataFrame(
    json_like_blocks,
    columns=["script_id", "length", "preview"]
)

print(f"Bloques candidatos a JSON embebido: {len(df_json_like_blocks)}")

df_json_like_blocks

Bloques candidatos a JSON embebido: 0


,script_id,length,preview


## 5. Extracción inicial de registros visibles en la aplicación RTVE

La aplicación `23fbuscador.rtve.es` sí contiene información documental directamente en el HTML.

En esta sección se comprueba si la tabla visible puede transformarse en un DataFrame estructurado.

El objetivo no es todavía construir el inventario completo de 167 documentos, sino validar:

- si existe una tabla HTML con documentos;
- qué columnas contiene;
- cuántas filas aparecen por defecto;
- si cada fila tiene enlace de detalle;
- si el HTML permite reconstruir la paginación;
- cómo se puede solicitar la vista de `200 / página`.

Si esta extracción funciona, el siguiente paso será automatizar la obtención de todos los registros.

In [21]:
# ============================================================
# 5. Extracción inicial de registros visibles en la app RTVE
# ============================================================

# ------------------------------------------------------------
# 5.1. Detección de tablas HTML en la aplicación
# ------------------------------------------------------------

app_tables = soup_rtve_app.find_all("table")

print(f"Número de tablas HTML encontradas en la app: {len(app_tables)}")

for i, table in enumerate(app_tables, start=1):
    rows = table.find_all("tr")
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]

    print(f"\nTabla {i}")
    print(f"- Filas detectadas: {len(rows)}")
    print(f"- Cabeceras: {headers}")

    print("- Primeras 3 filas:")
    for row in rows[:3]:
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["td", "th"])]
        print(cells)

Número de tablas HTML encontradas en la app: 1

Tabla 1
- Filas detectadas: 26
- Cabeceras: ['Título', 'Páginas', 'KB', 'Resumen', 'Palabras clave']
- Primeras 3 filas:
['Título', 'Páginas', 'KB', 'Resumen', 'Palabras clave']
['Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).', '3', '-', 'El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y certificaciones oficiales, aunque plagado de controversias por la interpretación y selección de testimonios, especialmente en re...', 'C/SG/2820/20-02-82 DTOR. Vista oral 2/81']
['Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).', '4', '-', 'Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, donde se analizaron declaraciones y diligencias relacionadas

In [22]:
# ------------------------------------------------------------
# 5.2. Extracción de filas de la tabla principal
# ------------------------------------------------------------

def extract_table_rows(table, base_url: str) -> pd.DataFrame:
    """
    Extrae filas de una tabla HTML y conserva texto, enlaces y posición.

    Parameters
    ----------
    table : bs4.element.Tag
        Tabla HTML.
    base_url : str
        URL base para convertir enlaces relativos en absolutos.

    Returns
    -------
    pd.DataFrame
        DataFrame con una fila por documento visible.
    """
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
    rows_data = []

    body_rows = table.find_all("tr")

    for row_idx, row in enumerate(body_rows, start=1):
        cells = row.find_all("td")

        # Saltar cabecera u otras filas sin celdas de datos
        if not cells:
            continue

        row_data = {
            "row_idx": row_idx,
            "n_cells": len(cells),
        }

        for col_idx, cell in enumerate(cells):
            col_name = headers[col_idx] if col_idx < len(headers) else f"col_{col_idx}"

            text = cell.get_text(" ", strip=True)
            link_tag = cell.find("a", href=True)
            href = urljoin(base_url, link_tag["href"]) if link_tag else None

            row_data[col_name] = text
            row_data[f"{col_name}_url"] = href

        rows_data.append(row_data)

    return pd.DataFrame(rows_data)


if len(app_tables) == 0:
    raise ValueError("No se han encontrado tablas en la aplicación RTVE.")

# En principio debería haber una tabla principal de documentos.
df_rtve_visible_rows_raw = extract_table_rows(app_tables[0], URL_RTVE_APP)

print(f"Filas de documentos visibles extraídas: {len(df_rtve_visible_rows_raw)}")

df_rtve_visible_rows_raw.head()

Filas de documentos visibles extraídas: 25


,row_idx,n_cells,Título,Título_url,Páginas,Páginas_url,KB,KB_url,Resumen,Resumen_url,Palabras clave,Palabras clave_url
0,2,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1860?page_size=25&page=1,3,None,-,None,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",None,C/SG/2820/20-02-82 DTOR. Vista oral 2/81,None
1,3,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1859?page_size=25&page=1,4,None,-,None,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,None,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,None
2,4,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1858?page_size=25&page=1,5,None,-,None,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,None,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
3,5,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1857?page_size=25&page=1,6,None,-,None,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,None,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
4,6,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1856?page_size=25&page=1,6,None,-,None,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la ses...,None,C/SG/3.249/26-02-82 SG Consejo Supremo de Justicia Militar,None


In [23]:
# ------------------------------------------------------------
# 5.3. Normalización inicial de columnas visibles
# ------------------------------------------------------------

def normalize_column_name(name: str) -> str:
    """
    Normaliza nombres de columnas procedentes del HTML.
    """
    if name is None:
        return ""

    name = str(name).strip().lower()
    name = (
        name.replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ñ", "n")
    )
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


df_rtve_visible_rows = df_rtve_visible_rows_raw.copy()
df_rtve_visible_rows.columns = [normalize_column_name(col) for col in df_rtve_visible_rows.columns]

print("Columnas normalizadas:")
print(df_rtve_visible_rows.columns.tolist())

df_rtve_visible_rows.head()

Columnas normalizadas:
['row_idx', 'n_cells', 'titulo', 'titulo_url', 'paginas', 'paginas_url', 'kb', 'kb_url', 'resumen', 'resumen_url', 'palabras_clave', 'palabras_clave_url']


,row_idx,n_cells,titulo,titulo_url,paginas,paginas_url,kb,kb_url,resumen,resumen_url,palabras_clave,palabras_clave_url
0,2,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1860?page_size=25&page=1,3,None,-,None,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",None,C/SG/2820/20-02-82 DTOR. Vista oral 2/81,None
1,3,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1859?page_size=25&page=1,4,None,-,None,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,None,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,None
2,4,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1858?page_size=25&page=1,5,None,-,None,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,None,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
3,5,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1857?page_size=25&page=1,6,None,-,None,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,None,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
4,6,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1856?page_size=25&page=1,6,None,-,None,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la ses...,None,C/SG/3.249/26-02-82 SG Consejo Supremo de Justicia Militar,None


In [24]:
# ------------------------------------------------------------
# 5.4. Inspección de enlaces de detalle en filas visibles
# ------------------------------------------------------------

url_columns = [col for col in df_rtve_visible_rows.columns if col.endswith("_url")]

print("Columnas con URLs detectadas:")
print(url_columns)

for col in url_columns:
    n_urls = df_rtve_visible_rows[col].notna().sum()
    print(f"{col}: {n_urls} URLs no nulas")

df_rtve_visible_rows[url_columns].head()

Columnas con URLs detectadas:
['titulo_url', 'paginas_url', 'kb_url', 'resumen_url', 'palabras_clave_url']
titulo_url: 25 URLs no nulas
paginas_url: 0 URLs no nulas
kb_url: 0 URLs no nulas
resumen_url: 0 URLs no nulas
palabras_clave_url: 0 URLs no nulas


,titulo_url,paginas_url,kb_url,resumen_url,palabras_clave_url
0,https://23fbuscador.rtve.es/document/ocr/1860?page_size=25&page=1,None,None,None,None
1,https://23fbuscador.rtve.es/document/ocr/1859?page_size=25&page=1,None,None,None,None
2,https://23fbuscador.rtve.es/document/ocr/1858?page_size=25&page=1,None,None,None,None
3,https://23fbuscador.rtve.es/document/ocr/1857?page_size=25&page=1,None,None,None,None
4,https://23fbuscador.rtve.es/document/ocr/1856?page_size=25&page=1,None,None,None,None


In [25]:
# ------------------------------------------------------------
# 5.5. Inspección de enlaces, formularios y controles de paginación
# ------------------------------------------------------------

# Enlaces de la app
app_links = []

for a in soup_rtve_app.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = urljoin(URL_RTVE_APP, a["href"])

    app_links.append({
        "text": text,
        "href": href,
    })

df_app_links = (
    pd.DataFrame(app_links)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Enlaces detectados en la app: {len(df_app_links)}")

# Mostrar enlaces que parezcan relacionados con paginación, tamaño de página o detalle
pagination_terms = ["página", "pagina", "siguiente", "anterior", "25", "50", "100", "200", "page", "limit", "size"]

mask_pagination = df_app_links["text"].str.lower().fillna("").apply(
    lambda x: any(term in x for term in pagination_terms)
) | df_app_links["href"].str.lower().fillna("").apply(
    lambda x: any(term in x for term in pagination_terms)
)

df_app_links_pagination = df_app_links[mask_pagination].copy()

df_app_links_pagination.head(100)

Enlaces detectados en la app: 28


,text,href
0,Siguiente,https://23fbuscador.rtve.es/?page_size=25&page=2
1,×,https://23fbuscador.rtve.es/?page_size=25
2,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1860?page_size=25&page=1
3,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1859?page_size=25&page=1
4,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1858?page_size=25&page=1
5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1857?page_size=25&page=1
6,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1856?page_size=25&page=1
7,Información integrada (11 de marzo de 1982).,https://23fbuscador.rtve.es/document/ocr/1855?page_size=25&page=1
8,Información integrada (16 de marzo de 1982).,https://23fbuscador.rtve.es/document/ocr/1854?page_size=25&page=1
9,Vista oral 2/81 del Consejo Supremo de Justicia Militar (2 de marzo de 1982).,https://23fbuscador.rtve.es/document/ocr/1853?page_size=25&page=1


In [26]:
# ------------------------------------------------------------
# 5.6. Inspección de formularios, inputs, selects y options
# ------------------------------------------------------------

forms_info = []

for form_idx, form in enumerate(soup_rtve_app.find_all("form"), start=1):
    form_action = urljoin(URL_RTVE_APP, form.get("action", ""))
    form_method = form.get("method", "GET").upper()

    inputs = []
    for input_tag in form.find_all("input"):
        inputs.append({
            "tag": "input",
            "type": input_tag.get("type"),
            "name": input_tag.get("name"),
            "value": input_tag.get("value"),
            "text": None,
        })

    for select_tag in form.find_all("select"):
        select_name = select_tag.get("name")

        for option_tag in select_tag.find_all("option"):
            inputs.append({
                "tag": "option",
                "type": "select_option",
                "name": select_name,
                "value": option_tag.get("value"),
                "text": option_tag.get_text(" ", strip=True),
            })

    forms_info.append({
        "form_idx": form_idx,
        "method": form_method,
        "action": form_action,
        "n_controls": len(inputs),
        "controls": inputs,
    })

print(f"Formularios detectados: {len(forms_info)}")

for form in forms_info:
    print(f"\nFormulario {form['form_idx']}")
    print(f"- Método: {form['method']}")
    print(f"- Action: {form['action']}")
    print(f"- Nº controles: {form['n_controls']}")

    for control in form["controls"][:30]:
        print(control)

Formularios detectados: 1

Formulario 1
- Método: GET
- Action: https://23fbuscador.rtve.es/
- Nº controles: 11
{'tag': 'input', 'type': 'text', 'name': 'q', 'value': '', 'text': None}
{'tag': 'input', 'type': 'text', 'name': None, 'value': None, 'text': None}
{'tag': 'input', 'type': 'hidden', 'name': 'q_all', 'value': '', 'text': None}
{'tag': 'input', 'type': 'text', 'name': None, 'value': None, 'text': None}
{'tag': 'input', 'type': 'hidden', 'name': 'q_any', 'value': '', 'text': None}
{'tag': 'input', 'type': 'text', 'name': None, 'value': None, 'text': None}
{'tag': 'input', 'type': 'hidden', 'name': 'q_not', 'value': '', 'text': None}
{'tag': 'option', 'type': 'select_option', 'name': 'page_size', 'value': '25', 'text': '25 / página'}
{'tag': 'option', 'type': 'select_option', 'name': 'page_size', 'value': '50', 'text': '50 / página'}
{'tag': 'option', 'type': 'select_option', 'name': 'page_size', 'value': '100', 'text': '100 / página'}
{'tag': 'option', 'type': 'select_option',

In [27]:
# ------------------------------------------------------------
# 5.7. Tabla de controles de formulario
# ------------------------------------------------------------

form_controls_rows = []

for form in forms_info:
    for control in form["controls"]:
        form_controls_rows.append({
            "form_idx": form["form_idx"],
            "method": form["method"],
            "action": form["action"],
            "tag": control.get("tag"),
            "type": control.get("type"),
            "name": control.get("name"),
            "value": control.get("value"),
            "text": control.get("text"),
        })

df_form_controls = pd.DataFrame(form_controls_rows)

df_form_controls

,form_idx,method,action,tag,type,name,value,text
0,1,GET,https://23fbuscador.rtve.es/,input,text,q,,None
1,1,GET,https://23fbuscador.rtve.es/,input,text,None,None,None
2,1,GET,https://23fbuscador.rtve.es/,input,hidden,q_all,,None
3,1,GET,https://23fbuscador.rtve.es/,input,text,None,None,None
4,1,GET,https://23fbuscador.rtve.es/,input,hidden,q_any,,None
5,1,GET,https://23fbuscador.rtve.es/,input,text,None,None,None
6,1,GET,https://23fbuscador.rtve.es/,input,hidden,q_not,,None
7,1,GET,https://23fbuscador.rtve.es/,option,select_option,page_size,25,25 / página
8,1,GET,https://23fbuscador.rtve.es/,option,select_option,page_size,50,50 / página
9,1,GET,https://23fbuscador.rtve.es/,option,select_option,page_size,100,100 / página


## 6. Construcción inicial del inventario RTVE con `page_size=200`

La aplicación RTVE permite modificar el número de resultados por página mediante el parámetro GET `page_size`.

En la inspección anterior se ha detectado que el formulario contiene las opciones:

- `25 / página`
- `50 / página`
- `100 / página`
- `200 / página`

Como el buscador indica que existen 167 resultados, solicitar `page_size=200` debería permitir obtener todos los documentos en una única página.

En esta sección se prueba esa hipótesis y se construye una primera versión del inventario RTVE con:

- identificador del documento;
- título;
- número de páginas;
- resumen;
- palabras clave;
- URL de detalle.

Todavía no se extrae el texto completo ni el PDF. Primero se valida que el inventario de metadatos se puede obtener correctamente.

In [28]:
# ============================================================
# 6. Construcción inicial del inventario RTVE con page_size=200
# ============================================================

from urllib.parse import urlparse, parse_qs

RTVE_PAGE_SIZE = 200

rtve_inventory_url = f"{URL_RTVE_APP}?page_size={RTVE_PAGE_SIZE}"

response_rtve_200 = fetch_response(rtve_inventory_url)
html_rtve_200 = response_rtve_200.text
soup_rtve_200 = BeautifulSoup(html_rtve_200, "html.parser")

print("Página RTVE con page_size=200 descargada correctamente.")
print(f"URL: {rtve_inventory_url}")
print(f"Status code: {response_rtve_200.status_code}")
print(f"Content-Type: {response_rtve_200.headers.get('Content-Type')}")
print(f"Número de caracteres del HTML: {len(html_rtve_200):,}")

text_rtve_200 = soup_rtve_200.get_text(" ", strip=True)

for term in ["167 resultados", "Página 1 de 1", "Página 1 de", "Texto completo", "Resumen", "Palabras clave"]:
    print(f"'{term}': {term.lower() in text_rtve_200.lower()}")

Página RTVE con page_size=200 descargada correctamente.
URL: https://23fbuscador.rtve.es/?page_size=200
Status code: 200
Content-Type: text/html; charset=utf-8
Número de caracteres del HTML: 203,248
'167 resultados': True
'Página 1 de 1': True
'Página 1 de': True
'Texto completo': False
'Resumen': True
'Palabras clave': True


In [29]:
# ------------------------------------------------------------
# 6.1. Extracción de la tabla con page_size=200
# ------------------------------------------------------------

tables_rtve_200 = soup_rtve_200.find_all("table")

print(f"Número de tablas encontradas con page_size=200: {len(tables_rtve_200)}")

if len(tables_rtve_200) == 0:
    raise ValueError("No se ha encontrado ninguna tabla en la página con page_size=200.")

for i, table in enumerate(tables_rtve_200, start=1):
    rows = table.find_all("tr")
    headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]

    print(f"\nTabla {i}")
    print(f"- Filas detectadas: {len(rows)}")
    print(f"- Cabeceras: {headers}")

df_rtve_200_raw = extract_table_rows(tables_rtve_200[0], URL_RTVE_APP)

print(f"\nFilas de documentos extraídas: {len(df_rtve_200_raw)}")

df_rtve_200_raw.head()

Número de tablas encontradas con page_size=200: 1

Tabla 1
- Filas detectadas: 168
- Cabeceras: ['Título', 'Páginas', 'KB', 'Resumen', 'Palabras clave']

Filas de documentos extraídas: 167


,row_idx,n_cells,Título,Título_url,Páginas,Páginas_url,KB,KB_url,Resumen,Resumen_url,Palabras clave,Palabras clave_url
0,2,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,3,None,-,None,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",None,C/SG/2820/20-02-82 DTOR. Vista oral 2/81,None
1,3,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,4,None,-,None,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,None,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,None
2,4,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,5,None,-,None,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,None,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
3,5,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,6,None,-,None,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,None,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,None
4,6,5,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1856?page_size=200&page=1,6,None,-,None,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la ses...,None,C/SG/3.249/26-02-82 SG Consejo Supremo de Justicia Militar,None


In [30]:
# ------------------------------------------------------------
# 6.2. Normalización del inventario RTVE
# ------------------------------------------------------------

def extract_rtve_document_id(url: str) -> str | None:
    """
    Extrae el identificador numérico de una URL de detalle RTVE.

    Ejemplo:
    https://23fbuscador.rtve.es/document/ocr/1860?page_size=25&page=1
    -> 1860
    """
    if pd.isna(url) or url is None:
        return None

    match = re.search(r"/document/ocr/(\d+)", str(url))
    return match.group(1) if match else None


df_rtve_inventory_work = df_rtve_200_raw.copy()
df_rtve_inventory_work.columns = [normalize_column_name(col) for col in df_rtve_inventory_work.columns]

df_rtve_inventory_work["source_document_id"] = df_rtve_inventory_work["titulo_url"].apply(extract_rtve_document_id)

df_rtve_inventory_work["doc_id"] = df_rtve_inventory_work["source_document_id"].apply(
    lambda x: f"rtve_{x}" if pd.notna(x) else None
)

# Normalizar páginas
df_rtve_inventory_work["pages"] = pd.to_numeric(
    df_rtve_inventory_work["paginas"],
    errors="coerce"
).astype("Int64")

# Construir inventario inicial
df_inventory_rtve = pd.DataFrame({
    "doc_id": df_rtve_inventory_work["doc_id"],
    "source": "rtve_buscador",
    "source_document_id": df_rtve_inventory_work["source_document_id"],
    "title": df_rtve_inventory_work["titulo"],
    "pages": df_rtve_inventory_work["pages"],
    "summary": df_rtve_inventory_work["resumen"],
    "keywords": df_rtve_inventory_work["palabras_clave"],
    "detail_url": df_rtve_inventory_work["titulo_url"],
    "pdf_url": None,
})

# Reordenar columnas: primero las previstas, luego identificador auxiliar
df_inventory_rtve = df_inventory_rtve[
    [
        "doc_id",
        "source",
        "source_document_id",
        "title",
        "pages",
        "summary",
        "keywords",
        "detail_url",
        "pdf_url",
    ]
]

print(f"Documentos en inventario RTVE: {len(df_inventory_rtve)}")
print(f"doc_id únicos: {df_inventory_rtve['doc_id'].nunique()}")
print(f"Títulos únicos: {df_inventory_rtve['title'].nunique()}")

df_inventory_rtve.head()

Documentos en inventario RTVE: 167
doc_id únicos: 167
Títulos únicos: 159


,doc_id,source,source_document_id,title,pages,summary,keywords,detail_url,pdf_url
0,rtve_1860,rtve_buscador,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,None
1,rtve_1859,rtve_buscador,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,None
2,rtve_1858,rtve_buscador,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,None
3,rtve_1857,rtve_buscador,1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,6,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,None
4,rtve_1856,rtve_buscador,1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,6,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la ses...,C/SG/3.249/26-02-82 SG Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1856?page_size=200&page=1,None


In [31]:
# ------------------------------------------------------------
# 6.3. Validaciones básicas del inventario RTVE
# ------------------------------------------------------------

expected_rtve_results = 167

validation_results = {
    "n_rows": len(df_inventory_rtve),
    "expected_rows": expected_rtve_results,
    "matches_expected_rows": len(df_inventory_rtve) == expected_rtve_results,
    "n_unique_doc_id": df_inventory_rtve["doc_id"].nunique(),
    "doc_id_all_present": df_inventory_rtve["doc_id"].notna().all(),
    "n_missing_title": df_inventory_rtve["title"].isna().sum(),
    "n_missing_pages": df_inventory_rtve["pages"].isna().sum(),
    "n_missing_summary": df_inventory_rtve["summary"].isna().sum(),
    "n_missing_keywords": df_inventory_rtve["keywords"].isna().sum(),
    "n_missing_detail_url": df_inventory_rtve["detail_url"].isna().sum(),
}

df_inventory_validation = pd.DataFrame(
    validation_results.items(),
    columns=["check", "value"]
)

df_inventory_validation

,check,value
0,n_rows,167
1,expected_rows,167
2,matches_expected_rows,True
3,n_unique_doc_id,167
4,doc_id_all_present,True
5,n_missing_title,0
6,n_missing_pages,0
7,n_missing_summary,0
8,n_missing_keywords,0
9,n_missing_detail_url,0


In [32]:
# ------------------------------------------------------------
# 6.4. Revisión de duplicados
# ------------------------------------------------------------

duplicated_doc_ids = df_inventory_rtve[df_inventory_rtve["doc_id"].duplicated(keep=False)]
duplicated_titles = df_inventory_rtve[df_inventory_rtve["title"].duplicated(keep=False)]

print(f"Filas con doc_id duplicado: {len(duplicated_doc_ids)}")
print(f"Filas con título duplicado: {len(duplicated_titles)}")

if len(duplicated_doc_ids) > 0:
    display(duplicated_doc_ids.sort_values("doc_id"))

if len(duplicated_titles) > 0:
    display(duplicated_titles.sort_values("title").head(20))

Filas con doc_id duplicado: 0
Filas con título duplicado: 12


,doc_id,source,source_document_id,title,pages,summary,keywords,detail_url,pdf_url
129,rtve_1731,rtve_buscador,1731,RESERVADO: comunicación procesamiento implicado.,1,"El documento oficial del Consejo Supremo de Justicia Militar, fechado el 17 de junio de 1981, remite al Ministro de ...",CONSEJO SUPREMO DE JUSTICIA MILITAR RELATORIA DE EJERCITO art.555 del Código de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1731?page_size=200&page=1,None
130,rtve_1730,rtve_buscador,1730,RESERVADO: comunicación procesamiento implicado.,1,"El documento es una comunicación oficial del Consejo Supremo de Justicia Militar, fechada el 12 de mayo de 1981, dir...",CONSEJO SUPREMO DE JUSTICIA MILITAR artº 555 del Código de Justicia Militar Consejo Reunido en Sala de Justicia,https://23fbuscador.rtve.es/document/ocr/1730?page_size=200&page=1,None
131,rtve_1729,rtve_buscador,1729,RESERVADO: comunicación procesamiento implicado.,1,La página 1 contiene una relatoría del Ejército dirigida al Ministro de Defensa en cumplimiento del artículo 555 del...,CONSEJO SUPREMO DE JUSTICIA MILITAR art. 555 del Código de Justicia Militar Consejo Reunido en Sala de Justicia,https://23fbuscador.rtve.es/document/ocr/1729?page_size=200&page=1,None
132,rtve_1728,rtve_buscador,1728,RESERVADO: comunicación procesamiento implicado.,1,"El documento es una comunicación formal del Consejo Supremo de Justicia Militar al Ministro de Defensa, fechada el 1...",CONSEJO SUPREMO DE JUSTICIA MILITAR RELATORIA DE EJERCITO art. 555 del Código de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1728?page_size=200&page=1,None
133,rtve_1727,rtve_buscador,1727,RESERVADO: comunicación procesamiento implicado.,1,"El documento es una comunicación oficial del Consejo Supremo de Justicia Militar, fechada el 14 de abril de 1981, di...",CONSEJO SUPREMO DE JUSTICIA MILITAR RELATORIA DE EJERCITO artículo 555 del Código de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1727?page_size=200&page=1,None
134,rtve_1726,rtve_buscador,1726,RESERVADO: comunicación procesamiento implicado.,1,"El documento es una comunicación oficial del Consejo Supremo de Justicia Militar, fechada el 10 de abril de 1981, di...",CONSEJO SUPREMO DE JUSTICIA MILITAR RELATORIA DE EJERCITO Artículo 555 del Código de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1726?page_size=200&page=1,None
136,rtve_1724,rtve_buscador,1724,RESERVADO: oficio dando cuenta toma de declaración.,1,"El documento oficial del Consejo Supremo de Justicia Militar, fechado el 13 de marzo de 1981, relata la instrucción ...",Consejo Supremo de Justicia Militar Juzgado Especial Causa s/n,https://23fbuscador.rtve.es/document/ocr/1724?page_size=200&page=1,None
138,rtve_1722,rtve_buscador,1722,RESERVADO: oficio dando cuenta toma de declaración.,1,"El informe del Consejo Supremo de Justicia Militar detalla los incidentes de los días 23 y 24 de febrero de 1981, in...",Consejo Supremo de Justicia Militar JUZGADO ESPECIAL Causa s/n,https://23fbuscador.rtve.es/document/ocr/1722?page_size=200&page=1,None
127,rtve_1733,rtve_buscador,1733,SECRETO: comunicación procesamiento implicado.,1,"El comunicado oficial del Ministerio de Defensa, fechado el 22 de abril de 1981 y dirigido al Almirante Jefe del Est...",MINISTERIO DE DEFENSA GABINETE DEL MINISTRO S.E.,https://23fbuscador.rtve.es/document/ocr/1733?page_size=200&page=1,None
128,rtve_1732,rtve_buscador,1732,SECRETO: comunicación procesamiento implicado.,1,"La página 1 contiene una comunicación oficial del Ministerio de Defensa, dirigida al Teniente General Jefe del Estad...",MINISTERIO DE DEFENSA GABINETE DEL MINISTRO Excmo. Sr. General de División don LUIS TORRES ROJAS,https://23fbuscador.rtve.es/document/ocr/1732?page_size=200&page=1,None


In [33]:
# ------------------------------------------------------------
# 6.5. Diferencia entre títulos repetidos y duplicados reales
# ------------------------------------------------------------

exact_duplicates = df_inventory_rtve[df_inventory_rtve.duplicated(keep=False)]
duplicated_detail_urls = df_inventory_rtve[df_inventory_rtve["detail_url"].duplicated(keep=False)]

print(f"Filas completamente duplicadas: {len(exact_duplicates)}")
print(f"Filas con detail_url duplicada: {len(duplicated_detail_urls)}")
print(f"Filas con título duplicado: {len(duplicated_titles)}")

if len(duplicated_titles) > 0:
    print(
        "Hay títulos repetidos, pero no se eliminan porque pueden corresponder "
        "a documentos distintos con distinto doc_id, URL, resumen o palabras clave."
    )

Filas completamente duplicadas: 0
Filas con detail_url duplicada: 0
Filas con título duplicado: 12
Hay títulos repetidos, pero no se eliminan porque pueden corresponder a documentos distintos con distinto doc_id, URL, resumen o palabras clave.


In [34]:
# ------------------------------------------------------------
# 6.6. Guardado del inventario RTVE
# ------------------------------------------------------------

can_save_inventory = (
    len(df_inventory_rtve) > 0
    and df_inventory_rtve["doc_id"].notna().all()
    and df_inventory_rtve["doc_id"].nunique() == len(df_inventory_rtve)
)

if can_save_inventory:
    df_inventory_rtve.to_csv(INVENTORY_RTVE_PATH, index=False, encoding="utf-8")
    print(f"Inventario RTVE guardado en: {INVENTORY_RTVE_PATH.relative_to(PROJECT_ROOT)}")
else:
    print("No se guarda el inventario porque no pasa las validaciones mínimas.")

Inventario RTVE guardado en: data/interim/inventory_rtve.csv


### 6.7. Conclusión provisional del inventario RTVE

La extracción con `page_size=200` permite obtener los 167 documentos indicados por el Buscador RTVE en una única página.

El inventario RTVE se ha construido correctamente con identificadores únicos (`doc_id`) y sin valores faltantes en los campos principales: título, páginas, resumen, palabras clave y URL de detalle.

Se han detectado títulos repetidos, pero no se consideran duplicados reales porque cada registro mantiene un identificador y una URL de detalle distintos. Estos títulos repetidos corresponden a documentos diferentes con denominaciones genéricas o similares.

Por tanto, el criterio de unicidad del inventario no será el título, sino el `doc_id` y la `detail_url`.

El siguiente paso será acceder a las páginas de detalle de cada documento para extraer el texto completo y comprobar si existe algún enlace o identificador del original digitalizado. Si esas páginas permiten obtener también el PDF final de cada documento, el corpus RTVE podrá construirse íntegramente desde la fuente principal, relacionando inventario, texto y PDF mediante `doc_id`.

## 7. Exploración de páginas de detalle RTVE

El inventario RTVE ya contiene una URL de detalle por documento en la columna `detail_url`.

Estas páginas son importantes porque, según la inspección visual del navegador, contienen:

- el texto completo del documento;
- el visor del original digitalizado;
- posibles enlaces o identificadores que pueden ayudar a relacionar cada documento con su PDF.

En esta sección se inspeccionará una página de detalle concreta antes de automatizar la extracción para los 167 documentos.

El objetivo es responder:

1. ¿El texto completo aparece directamente en el HTML?
2. ¿En qué etiqueta o bloque HTML aparece?
3. ¿Existe algún enlace directo o indirecto al original digitalizado?
4. ¿Podemos construir después `document_texts_rtve.csv` usando `doc_id` como clave?

In [35]:
# ============================================================
# 7. Exploración de páginas de detalle RTVE
# ============================================================

# Seleccionamos el primer documento del inventario como caso de prueba.
sample_doc = df_inventory_rtve.iloc[0].to_dict()

sample_doc_id = sample_doc["doc_id"]
sample_title = sample_doc["title"]
sample_detail_url = sample_doc["detail_url"]

print("Documento de prueba seleccionado:")
print(f"doc_id: {sample_doc_id}")
print(f"title: {sample_title}")
print(f"detail_url: {sample_detail_url}")

Documento de prueba seleccionado:
doc_id: rtve_1860
title: Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).
detail_url: https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1


In [36]:
# ------------------------------------------------------------
# 7.1. Descarga de la página de detalle
# ------------------------------------------------------------

response_detail = fetch_response(sample_detail_url)
html_detail = response_detail.text
soup_detail = BeautifulSoup(html_detail, "html.parser")

print("Página de detalle descargada correctamente.")
print(f"Status code: {response_detail.status_code}")
print(f"Content-Type: {response_detail.headers.get('Content-Type')}")
print(f"Número de caracteres del HTML: {len(html_detail):,}")
print(f"Título HTML: {soup_detail.title.get_text(strip=True) if soup_detail.title else None}")

Página de detalle descargada correctamente.
Status code: 200
Content-Type: text/html; charset=utf-8
Número de caracteres del HTML: 12,165
Título HTML: Detalle documento


In [37]:
# ------------------------------------------------------------
# 7.2. Comprobación de términos clave en la página de detalle
# ------------------------------------------------------------

detail_text_visible = soup_detail.get_text(" ", strip=True)

detail_search_terms = [
    "TEXTO COMPLETO",
    "Texto completo",
    "ORIGINAL DIGITALIZADO",
    "Original digitalizado",
    "AUDIO",
    "VIDEO",
    "C/SG/2820/20-02-82",
    "NOTA INFORMATIVA",
    "pdf",
    "download",
    "visor",
]

for term in detail_search_terms:
    found = term.lower() in detail_text_visible.lower()
    print(f"'{term}': {found}")

print()
print("Primeros 1.500 caracteres de texto visible:")
print(detail_text_visible[:1500])

'TEXTO COMPLETO': True
'Texto completo': True
'ORIGINAL DIGITALIZADO': True
'Original digitalizado': True
'AUDIO': True
'VIDEO': True
'C/SG/2820/20-02-82': True
'NOTA INFORMATIVA': True
'pdf': False
'download': False
'visor': False

Primeros 1.500 caracteres de texto visible:
Detalle documento 23F Documentos Desclasificados Detalle de documento Volver al listado Anterior Documento 1 de 167 Siguiente Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982). ocr Tipo: ocr Estado: ok Páginas: 3 KB: - Modelo: mistral-ocr-latest Proveedor: mistral Resumen El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y certificaciones oficiales, aunque plagado de controversias por la interpretación y selección de testimonios, especialmente en relación con figuras clave como el Teniente Coronel Tejero y autoridades civiles. Surgieron tensiones notables entre la Fi

In [38]:
# ------------------------------------------------------------
# 7.3. Inspección de bloques largos de texto
# ------------------------------------------------------------

def describe_html_element(tag) -> dict:
    """
    Devuelve información resumida de un elemento HTML.
    """
    text = tag.get_text("\n", strip=True)
    return {
        "tag": tag.name,
        "id": tag.get("id"),
        "class": " ".join(tag.get("class", [])) if tag.get("class") else None,
        "text_length": len(text),
        "text_preview": text[:500],
    }


candidate_text_blocks = []

# Buscamos etiquetas que suelen contener texto largo o estructurado.
for tag in soup_detail.find_all(["pre", "textarea", "div", "section", "article", "main", "p"]):
    text = tag.get_text("\n", strip=True)

    if len(text) >= 300:
        candidate_text_blocks.append(describe_html_element(tag))

df_detail_text_blocks = (
    pd.DataFrame(candidate_text_blocks)
    .drop_duplicates(subset=["tag", "id", "class", "text_preview"])
    .sort_values("text_length", ascending=False)
    .reset_index(drop=True)
)

print(f"Bloques candidatos con texto largo: {len(df_detail_text_blocks)}")

df_detail_text_blocks.head(20)

Bloques candidatos con texto largo: 10


,tag,id,class,text_length,text_preview
0,main,None,container,5955,Volver al listado\nAnterior\nDocumento 1 de 167\nSiguiente\nVista oral 2/81 del Consejo Supremo de Justicia Militar ...
1,section,None,detail-section detail-columns,3987,Texto completo\nC/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SES...
2,article,None,detail-column,3949,Texto completo\nC/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SES...
3,pre,None,text-box text-box-large,3934,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIE...
4,section,None,detail-section,998,Resumen\nEl juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras s...
5,p,None,text-box,990,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ..."
6,section,None,detail-section,703,Palabras clave\nPersonas\n10\nNo consta\nLuis Arana Lorite:Teniente Coronel\nManuel Miler Hidalgo:Teniente Coronel\n...
7,div,None,tag-grid,688,Personas\n10\nNo consta\nLuis Arana Lorite:Teniente Coronel\nManuel Miler Hidalgo:Teniente Coronel\nTEJERO:Teniente ...
8,article,None,tag-group-card tag-group-people,312,Personas\n10\nNo consta\nLuis Arana Lorite:Teniente Coronel\nManuel Miler Hidalgo:Teniente Coronel\nTEJERO:Teniente ...
9,div,None,chips-wrap,300,No consta\nLuis Arana Lorite:Teniente Coronel\nManuel Miler Hidalgo:Teniente Coronel\nTEJERO:Teniente Coronel\nSR. C...


In [39]:
# ------------------------------------------------------------
# 7.4. Inspección de visores, iframes, embeds y enlaces
# ------------------------------------------------------------

embedded_resources = []

# iframes
for tag in soup_detail.find_all("iframe"):
    src = tag.get("src")
    embedded_resources.append({
        "tag": "iframe",
        "url": urljoin(sample_detail_url, src) if src else None,
        "text": tag.get_text(" ", strip=True),
    })

# embeds
for tag in soup_detail.find_all("embed"):
    src = tag.get("src")
    embedded_resources.append({
        "tag": "embed",
        "url": urljoin(sample_detail_url, src) if src else None,
        "text": tag.get_text(" ", strip=True),
    })

# objects
for tag in soup_detail.find_all("object"):
    data = tag.get("data")
    embedded_resources.append({
        "tag": "object",
        "url": urljoin(sample_detail_url, data) if data else None,
        "text": tag.get_text(" ", strip=True),
    })

# enlaces
for tag in soup_detail.find_all("a", href=True):
    href = urljoin(sample_detail_url, tag["href"])
    text = tag.get_text(" ", strip=True)

    if any(term in href.lower() for term in ["pdf", "download", "media", "document", "ocr", "file", "original", "viewer"]):
        embedded_resources.append({
            "tag": "a",
            "url": href,
            "text": text,
        })

df_embedded_resources = (
    pd.DataFrame(embedded_resources, columns=["tag", "url", "text"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Recursos embebidos o enlaces candidatos detectados: {len(df_embedded_resources)}")

df_embedded_resources

Recursos embebidos o enlaces candidatos detectados: 3


,tag,url,text
0,iframe,https://23fbuscador.rtve.es/document/ocr/1860/original,
1,a,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,Siguiente
2,a,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1#page-top,↑


In [40]:
# ------------------------------------------------------------
# 7.5. Extracción directa del texto completo y del visor original
# ------------------------------------------------------------

def extract_text_and_original_from_detail(detail_url: str) -> dict:
    """
    Extrae el texto completo y la URL del visor/original digitalizado
    desde una página de detalle de RTVE.

    Parameters
    ----------
    detail_url : str
        URL de detalle del documento en el Buscador RTVE.

    Returns
    -------
    dict
        Diccionario con texto completo, métricas básicas y URL del visor original.
    """
    response = fetch_response(detail_url)
    soup = BeautifulSoup(response.text, "html.parser")

    # Texto completo: bloque principal localizado en la inspección manual.
    text_block = soup.find("pre", class_="text-box text-box-large")

    text_full = text_block.get_text("\n", strip=True) if text_block else None

    # Visor/original digitalizado: normalmente aparece como iframe.
    original_iframe = soup.find("iframe")
    original_viewer_url = None

    if original_iframe and original_iframe.get("src"):
        original_viewer_url = urljoin(detail_url, original_iframe["src"])

    return {
        "text_full": text_full,
        "text_length_chars": len(text_full) if text_full else 0,
        "text_length_words": len(text_full.split()) if text_full else 0,
        "text_extraction_ok": text_full is not None and len(text_full.strip()) > 0,
        "original_viewer_url": original_viewer_url,
    }


sample_extraction = extract_text_and_original_from_detail(sample_detail_url)

print("Extracción del documento de prueba:")
print(f"text_extraction_ok: {sample_extraction['text_extraction_ok']}")
print(f"text_length_chars: {sample_extraction['text_length_chars']}")
print(f"text_length_words: {sample_extraction['text_length_words']}")
print(f"original_viewer_url: {sample_extraction['original_viewer_url']}")
print()
print("Primeros 1.000 caracteres del texto completo:")
print(sample_extraction["text_full"][:1000])

Extracción del documento de prueba:
text_extraction_ok: True
text_length_chars: 3934
text_length_words: 640
original_viewer_url: https://23fbuscador.rtve.es/document/ocr/1860/original

Primeros 1.000 caracteres del texto completo:
C/SG/2820/20-02-82
DTOR.

NOTA INFORMATIVA

ASUNTO: Vista oral 2/81

1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82

- Solo ha tenido lugar la sesión de la mañana. Empezó a las 10,06 horas.

- Durante la sesión han tenido lugar, a petición del Sr. Fiscal las declaraciones siguientes:

. Parcial de Teniente Coronel D. Luis Arana Lorite (Ayte. Gral. Lluch).

. Parcial de Teniente Coronel D. Manuel Miler Hidalgo (Ayte. Gral. Esquivias).

. 1ª, 2ª, 3ª y 4ª del Teniente Coronel TEJERO.

. Careo Teniente Coronel TEJERO - SR. CARRES.

- Descanso de 11,50 horas a 12,13 horas.

. Careo Teniente Coronel TEJERO - Capitán GOMEZ IGLESIAS

. Certificación del Presidente del Congreso Sr. LAVILLA.

. Certificación sobre la situación militar del TCOL.TEJERO.

. 1ª y 

In [41]:
# ------------------------------------------------------------
# 7.6. Validación del tipo de recurso original
# ------------------------------------------------------------

def inspect_resource_url(url: str, timeout: int = 20) -> dict:
    """
    Inspecciona una URL sin descargar completamente el archivo.

    Se usa para comprobar si el recurso original es un PDF, HTML,
    audio, vídeo u otro tipo de contenido.

    Parameters
    ----------
    url : str
        URL del recurso.
    timeout : int
        Tiempo máximo de espera.

    Returns
    -------
    dict
        Información básica del recurso.
    """
    result = {
        "url": url,
        "final_url": None,
        "status_code": None,
        "ok": False,
        "content_type": None,
        "content_length": None,
        "is_pdf": False,
        "error": None,
    }

    if pd.isna(url) or url is None:
        result["error"] = "URL vacía"
        return result

    try:
        # Primero intentamos HEAD para evitar descargar el archivo.
        response = requests.head(
            url,
            headers=REQUEST_HEADERS,
            timeout=timeout,
            allow_redirects=True,
        )

        # Algunos servidores no responden bien a HEAD.
        if response.status_code >= 400 or not response.headers.get("Content-Type"):
            response = requests.get(
                url,
                headers=REQUEST_HEADERS,
                timeout=timeout,
                stream=True,
                allow_redirects=True,
            )

        content_type = response.headers.get("Content-Type")
        content_length = response.headers.get("Content-Length")
        final_url = response.url

        result.update({
            "final_url": final_url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": content_type,
            "content_length": content_length,
            "is_pdf": (
                content_type is not None
                and "application/pdf" in content_type.lower()
            ) or str(final_url).lower().endswith(".pdf"),
        })

    except Exception as exc:
        result["error"] = str(exc)

    return result


sample_resource_info = inspect_resource_url(sample_extraction["original_viewer_url"])

sample_resource_info

{'url': 'https://23fbuscador.rtve.es/document/ocr/1860/original',
 'final_url': 'https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrero_de_1982.pdf',
 'status_code': 200,
 'ok': True,
 'content_type': 'application/pdf',
 'content_length': '771201',
 'is_pdf': True,
 'error': None}

### 7.7. Conclusión sobre la página de detalle

La página de detalle de RTVE contiene el texto completo directamente en el HTML.

El bloque relevante se encuentra en una etiqueta:

`<pre class="text-box text-box-large">`

Esto permite extraer el texto completo sin OCR y sin descargar previamente los PDFs.

Además, la página incluye un `iframe` hacia el original digitalizado, con una URL del tipo:

`https://23fbuscador.rtve.es/document/ocr/{source_document_id}/original`

En el documento de prueba, esta URL redirige a un recurso con `Content-Type = application/pdf`, por lo que puede utilizarse para obtener el enlace final al PDF.

La estrategia queda definida así:

- `inventory_rtve.csv`: metadatos del listado RTVE.
- `document_texts_rtve.csv`: texto completo extraído desde las páginas de detalle.
- `original_url`: enlace interno al original detectado en la página de detalle.
- `original_final_url`: URL final tras resolver posibles redirecciones.
- `pdf_url`: URL final del PDF cuando el recurso original tenga `Content-Type = application/pdf`.

Con este resultado, la página de detalle de RTVE pasa a ser suficiente para obtener tanto el texto completo como el enlace final al PDF. Por tanto, el corpus base puede construirse desde la fuente principal de RTVE sin depender de una fuente adicional para localizar los PDFs.

## 8. Extracción de textos completos y PDFs para todos los documentos RTVE

Una vez validada la página de detalle con un documento de prueba, se aplica la misma lógica a los 167 documentos del inventario RTVE.

Para cada documento se extraerá:

- `doc_id`;
- `source_document_id`;
- `detail_url`;
- texto completo;
- longitud en caracteres;
- longitud aproximada en palabras;
- estado de extracción;
- URL interna del original (`original_url`);
- URL final del original tras redirección (`original_final_url`);
- tipo de contenido del original;
- indicador de si el original es PDF;
- `pdf_url`, cuando el original validado sea efectivamente un PDF.

El resultado principal se guardará en:

`data/interim/document_texts_rtve.csv`

Además, se actualizará el inventario RTVE con `pdf_url` cuando el recurso original sea un PDF validado.

Si la extracción se completa correctamente para los 167 documentos, RTVE proporcionará por sí sola los tres elementos necesarios para el corpus base: metadatos, texto completo y PDF asociado.

In [42]:
# ============================================================
# 8. Extracción de textos completos para todos los documentos RTVE
# ============================================================

from time import sleep

def extract_document_detail(row: pd.Series, sleep_seconds: float = 0.15) -> dict:
    """
    Extrae texto completo y recurso original de un documento RTVE.

    Parameters
    ----------
    row : pd.Series
        Fila del inventario RTVE.
    sleep_seconds : float
        Pausa breve para no hacer peticiones demasiado agresivas.

    Returns
    -------
    dict
        Resultado de extracción para un documento.
    """
    doc_id = row.get("doc_id")
    source_document_id = row.get("source_document_id")
    detail_url = row.get("detail_url")

    result = {
        "doc_id": doc_id,
        "source_document_id": source_document_id,
        "detail_url": detail_url,
        "text_full": None,
        "text_length_chars": 0,
        "text_length_words": 0,
        "extraction_source": "rtve_detail_html_pre",
        "text_extraction_ok": False,
        "original_url": None,
        "original_final_url": None,
        "original_content_type": None,
        "original_content_length": None,
        "original_is_pdf": False,
        "pdf_url": None,
        "error": None,
    }

    try:
        response = fetch_response(detail_url)
        soup = BeautifulSoup(response.text, "html.parser")

        # Texto completo
        text_block = soup.find("pre", class_="text-box text-box-large")
        text_full = text_block.get_text("\n", strip=True) if text_block else None

        if text_full and text_full.strip():
            result["text_full"] = text_full
            result["text_length_chars"] = len(text_full)
            result["text_length_words"] = len(text_full.split())
            result["text_extraction_ok"] = True

        # Original digitalizado
        original_iframe = soup.find("iframe")
        original_url = None

        if original_iframe and original_iframe.get("src"):
            original_url = urljoin(detail_url, original_iframe["src"])
            result["original_url"] = original_url

            resource_info = inspect_resource_url(original_url)

            result["original_final_url"] = resource_info.get("final_url")
            result["original_content_type"] = resource_info.get("content_type")
            result["original_content_length"] = resource_info.get("content_length")
            result["original_is_pdf"] = resource_info.get("is_pdf", False)

            if result["original_is_pdf"]:
                result["pdf_url"] = result["original_final_url"] or original_url

        sleep(sleep_seconds)

    except Exception as exc:
        result["error"] = str(exc)

    return result


detail_results = []

for idx, row in df_inventory_rtve.iterrows():
    if (idx + 1) % 25 == 0 or idx == 0:
        print(f"Procesando documento {idx + 1}/{len(df_inventory_rtve)}")

    detail_results.append(extract_document_detail(row))

df_document_texts_rtve = pd.DataFrame(detail_results)

print("Extracción completada.")
print(f"Documentos procesados: {len(df_document_texts_rtve)}")
print(f"Textos extraídos correctamente: {df_document_texts_rtve['text_extraction_ok'].sum()}")
print(f"Originales detectados: {df_document_texts_rtve['original_url'].notna().sum()}")
print(f"Originales identificados como PDF: {df_document_texts_rtve['original_is_pdf'].sum()}")

df_document_texts_rtve.head()

Procesando documento 1/167
Procesando documento 25/167
Procesando documento 50/167
Procesando documento 75/167
Procesando documento 100/167
Procesando documento 125/167
Procesando documento 150/167
Extracción completada.
Documentos procesados: 167
Textos extraídos correctamente: 167
Originales detectados: 167
Originales identificados como PDF: 167


,doc_id,source_document_id,detail_url,text_full,text_length_chars,text_length_words,extraction_source,text_extraction_ok,original_url,original_final_url,original_content_type,original_content_length,original_is_pdf,pdf_url,error
0,rtve_1860,1860,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIE...,3934,640,rtve_detail_html_pre,True,https://23fbuscador.rtve.es/document/ocr/1860/original,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,application/pdf,771201,True,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,None
1,rtve_1859,1859,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.-...,6417,1018,rtve_detail_html_pre,True,https://23fbuscador.rtve.es/document/ocr/1859/original,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,application/pdf,979124,True,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,None
2,rtve_1858,1858,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## ...,8183,1347,rtve_detail_html_pre,True,https://23fbuscador.rtve.es/document/ocr/1858/original,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,application/pdf,1241724,True,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,None
3,rtve_1857,1857,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## ...,11151,1826,rtve_detail_html_pre,True,https://23fbuscador.rtve.es/document/ocr/1857/original,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,application/pdf,1680589,True,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,None
4,rtve_1856,1856,https://23fbuscador.rtve.es/document/ocr/1856?page_size=200&page=1,"C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral de la causa 2/81, del Consejo Supremo de Justici...",10124,1740,rtve_detail_html_pre,True,https://23fbuscador.rtve.es/document/ocr/1856/original,https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,application/pdf,973226,True,https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,None


In [43]:
# ------------------------------------------------------------
# 8.1. Validaciones de la extracción de textos
# ------------------------------------------------------------

text_extraction_validation = {
    "n_rows": len(df_document_texts_rtve),
    "expected_rows": len(df_inventory_rtve),
    "matches_inventory_rows": len(df_document_texts_rtve) == len(df_inventory_rtve),
    "n_unique_doc_id": df_document_texts_rtve["doc_id"].nunique(),
    "doc_id_all_present": df_document_texts_rtve["doc_id"].notna().all(),
    "n_text_ok": int(df_document_texts_rtve["text_extraction_ok"].sum()),
    "n_text_failed": int((~df_document_texts_rtve["text_extraction_ok"]).sum()),
    "n_original_url": int(df_document_texts_rtve["original_url"].notna().sum()),
    "n_original_pdf": int(df_document_texts_rtve["original_is_pdf"].sum()),
    "n_errors": int(df_document_texts_rtve["error"].notna().sum()),
    "min_text_length_chars": int(df_document_texts_rtve["text_length_chars"].min()),
    "median_text_length_chars": float(df_document_texts_rtve["text_length_chars"].median()),
    "max_text_length_chars": int(df_document_texts_rtve["text_length_chars"].max()),
}

df_text_extraction_validation = pd.DataFrame(
    text_extraction_validation.items(),
    columns=["check", "value"]
)

df_text_extraction_validation

,check,value
0,n_rows,167
1,expected_rows,167
2,matches_inventory_rows,True
3,n_unique_doc_id,167
4,doc_id_all_present,True
5,n_text_ok,167
6,n_text_failed,0
7,n_original_url,167
8,n_original_pdf,167
9,n_errors,0


In [44]:
# ------------------------------------------------------------
# 8.2. Revisión de documentos con extracción fallida
# ------------------------------------------------------------

df_failed_texts = df_document_texts_rtve[
    ~df_document_texts_rtve["text_extraction_ok"]
].copy()

print(f"Documentos con extracción de texto fallida: {len(df_failed_texts)}")

if len(df_failed_texts) > 0:
    display(df_failed_texts[["doc_id", "source_document_id", "detail_url", "error"]].head(20))

Documentos con extracción de texto fallida: 0


In [45]:
# ------------------------------------------------------------
# 8.3. Guardado de la tabla de textos completos
# ------------------------------------------------------------

can_save_texts = (
    len(df_document_texts_rtve) == len(df_inventory_rtve)
    and df_document_texts_rtve["doc_id"].nunique() == len(df_document_texts_rtve)
    and df_document_texts_rtve["doc_id"].notna().all()
)

if can_save_texts:
    df_document_texts_rtve.to_csv(DOCUMENT_TEXTS_RTVE_PATH, index=False, encoding="utf-8")
    print(f"Tabla de textos RTVE guardada en: {DOCUMENT_TEXTS_RTVE_PATH.relative_to(PROJECT_ROOT)}")
else:
    print("No se guarda la tabla de textos porque no pasa las validaciones mínimas.")

Tabla de textos RTVE guardada en: data/interim/document_texts_rtve.csv


In [46]:
# ------------------------------------------------------------
# 8.4. Actualización del inventario RTVE con pdf_url validado
# ------------------------------------------------------------

df_pdf_urls = df_document_texts_rtve[["doc_id", "pdf_url"]].copy()

df_inventory_rtve_updated = (
    df_inventory_rtve
    .drop(columns=["pdf_url"], errors="ignore")
    .merge(df_pdf_urls, on="doc_id", how="left")
)

# Reordenar columnas
df_inventory_rtve_updated = df_inventory_rtve_updated[
    [
        "doc_id",
        "source",
        "source_document_id",
        "title",
        "pages",
        "summary",
        "keywords",
        "detail_url",
        "pdf_url",
    ]
]

df_inventory_rtve_updated.to_csv(INVENTORY_RTVE_PATH, index=False, encoding="utf-8")

print(f"Inventario RTVE actualizado con pdf_url en: {INVENTORY_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Registros con pdf_url: {df_inventory_rtve_updated['pdf_url'].notna().sum()}")

df_inventory_rtve = df_inventory_rtve_updated.copy()

df_inventory_rtve.head()

Inventario RTVE actualizado con pdf_url en: data/interim/inventory_rtve.csv
Registros con pdf_url: 167


,doc_id,source,source_document_id,title,pages,summary,keywords,detail_url,pdf_url
0,rtve_1860,rtve_buscador,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
1,rtve_1859,rtve_buscador,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
2,rtve_1858,rtve_buscador,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
3,rtve_1857,rtve_buscador,1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,6,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
4,rtve_1856,rtve_buscador,1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,6,Resumen global del documento sobre la sesión del Consejo Supremo de Justicia Militar (26-02-1982):\n\nDurante la ses...,C/SG/3.249/26-02-82 SG Consejo Supremo de Justicia Militar,https://23fbuscador.rtve.es/document/ocr/1856?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...


In [47]:
# ------------------------------------------------------------
# 8.5. Normalización final de salidas: textos, PDFs e inventario
# ------------------------------------------------------------

# Validamos si todos los originales detectados son PDFs.
all_originals_are_pdf = df_document_texts_rtve["original_is_pdf"].all()
all_content_types_are_pdf = df_document_texts_rtve["original_content_type"].eq("application/pdf").all()
all_final_urls_match_pdf_url = df_document_texts_rtve["original_final_url"].eq(
    df_document_texts_rtve["pdf_url"]
).all()

print(f"Todos los originales son PDF: {all_originals_are_pdf}")
print(f"Todos los Content-Type son application/pdf: {all_content_types_are_pdf}")
print(f"original_final_url coincide con pdf_url en todos los casos: {all_final_urls_match_pdf_url}")

# ------------------------------------------------------------
# Tabla final de textos
# ------------------------------------------------------------
# Incluimos pdf_url por comodidad práctica: permite consultar el PDF asociado
# sin tener que hacer merge con el inventario.

TEXT_COLUMNS_FINAL = [
    "doc_id",
    "text_full",
    "text_length_chars",
    "text_length_words",
    "extraction_source",
    "text_extraction_ok",
    "pdf_url",
]

df_document_texts_rtve_final = df_document_texts_rtve[TEXT_COLUMNS_FINAL].copy()

# ------------------------------------------------------------
# Inventario final RTVE
# ------------------------------------------------------------
# El inventario mantiene metadatos documentales y pdf_url.
# No incluye texto completo para evitar una tabla pesada.

df_inventory_rtve_final = df_inventory_rtve.copy()

# ------------------------------------------------------------
# Manifiesto de PDFs
# ------------------------------------------------------------
# Esta tabla no se crea para análisis, sino para facilitar la descarga física
# posterior de los PDFs en data/raw/pdfs_rtve/.
#
# local_pdf_path define dónde debería guardarse cada PDF.
# download_ok y file_size_bytes se completarán cuando se descarguen los archivos.

def build_pdf_filename(row: pd.Series) -> str:
    """
    Construye un nombre de archivo estable para guardar cada PDF.

    Se usa doc_id para evitar problemas con títulos largos, duplicados,
    caracteres especiales o nombres no compatibles con el sistema de archivos.
    """
    return f"{row['doc_id']}.pdf"


df_pdf_manifest_rtve = df_document_texts_rtve[
    [
        "doc_id",
        "source_document_id",
        "pdf_url",
        "original_url",
        "original_content_length",
    ]
].copy()

df_pdf_manifest_rtve["local_pdf_path"] = df_pdf_manifest_rtve.apply(
    lambda row: str((RAW_PDFS_RTVE_DIR / build_pdf_filename(row)).relative_to(PROJECT_ROOT)),
    axis=1,
)

df_pdf_manifest_rtve = df_pdf_manifest_rtve.rename(
    columns={
        "original_content_length": "expected_size_bytes",
    }
)

df_pdf_manifest_rtve["expected_size_bytes"] = pd.to_numeric(
    df_pdf_manifest_rtve["expected_size_bytes"],
    errors="coerce"
).astype("Int64")

df_pdf_manifest_rtve["download_ok"] = False
df_pdf_manifest_rtve["file_size_bytes"] = pd.NA

df_pdf_manifest_rtve = df_pdf_manifest_rtve[
    [
        "doc_id",
        "source_document_id",
        "pdf_url",
        "original_url",
        "local_pdf_path",
        "expected_size_bytes",
        "download_ok",
        "file_size_bytes",
    ]
]

# ------------------------------------------------------------
# Guardado de salidas finales
# ------------------------------------------------------------

df_document_texts_rtve_final.to_csv(DOCUMENT_TEXTS_RTVE_PATH, index=False, encoding="utf-8")
df_inventory_rtve_final.to_csv(INVENTORY_RTVE_PATH, index=False, encoding="utf-8")
df_pdf_manifest_rtve.to_csv(PDF_MANIFEST_RTVE_PATH, index=False, encoding="utf-8")

print(f"Textos RTVE guardados en: {DOCUMENT_TEXTS_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Inventario RTVE guardado en: {INVENTORY_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Manifiesto de PDFs RTVE guardado en: {PDF_MANIFEST_RTVE_PATH.relative_to(PROJECT_ROOT)}")

print()
print("Dimensiones finales:")
print(f"- inventory_rtve: {df_inventory_rtve_final.shape}")
print(f"- document_texts_rtve: {df_document_texts_rtve_final.shape}")
print(f"- pdf_manifest_rtve: {df_pdf_manifest_rtve.shape}")

Todos los originales son PDF: True
Todos los Content-Type son application/pdf: True
original_final_url coincide con pdf_url en todos los casos: True
Textos RTVE guardados en: data/interim/document_texts_rtve.csv
Inventario RTVE guardado en: data/interim/inventory_rtve.csv
Manifiesto de PDFs RTVE guardado en: data/interim/pdf_manifest_rtve.csv

Dimensiones finales:
- inventory_rtve: (167, 9)
- document_texts_rtve: (167, 7)
- pdf_manifest_rtve: (167, 8)


In [48]:
# ------------------------------------------------------------
# 8.6. Validación final de consistencia entre salidas RTVE
# ------------------------------------------------------------

rtve_output_validation = {
    "inventory_rows": len(df_inventory_rtve_final),
    "texts_rows": len(df_document_texts_rtve_final),
    "pdf_manifest_rows": len(df_pdf_manifest_rtve),

    "same_number_of_rows": (
        len(df_inventory_rtve_final)
        == len(df_document_texts_rtve_final)
        == len(df_pdf_manifest_rtve)
    ),

    "inventory_unique_doc_id": df_inventory_rtve_final["doc_id"].nunique(),
    "texts_unique_doc_id": df_document_texts_rtve_final["doc_id"].nunique(),
    "pdf_manifest_unique_doc_id": df_pdf_manifest_rtve["doc_id"].nunique(),

    "all_texts_extracted": df_document_texts_rtve_final["text_extraction_ok"].all(),

    "inventory_all_pdf_url_present": df_inventory_rtve_final["pdf_url"].notna().all(),
    "texts_all_pdf_url_present": df_document_texts_rtve_final["pdf_url"].notna().all(),
    "manifest_all_pdf_url_present": df_pdf_manifest_rtve["pdf_url"].notna().all(),

    "inventory_pdf_urls_unique": df_inventory_rtve_final["pdf_url"].nunique() == len(df_inventory_rtve_final),
    "texts_pdf_urls_unique": df_document_texts_rtve_final["pdf_url"].nunique() == len(df_document_texts_rtve_final),
    "manifest_pdf_urls_unique": df_pdf_manifest_rtve["pdf_url"].nunique() == len(df_pdf_manifest_rtve),

    "manifest_all_local_paths_present": df_pdf_manifest_rtve["local_pdf_path"].notna().all(),
    "manifest_local_paths_unique": df_pdf_manifest_rtve["local_pdf_path"].nunique() == len(df_pdf_manifest_rtve),

    "manifest_download_ok_initially_false": (df_pdf_manifest_rtve["download_ok"] == False).all(),
}

df_rtve_output_validation = pd.DataFrame(
    rtve_output_validation.items(),
    columns=["check", "value"]
)

df_rtve_output_validation

,check,value
0,inventory_rows,167
1,texts_rows,167
2,pdf_manifest_rows,167
3,same_number_of_rows,True
4,inventory_unique_doc_id,167
5,texts_unique_doc_id,167
6,pdf_manifest_unique_doc_id,167
7,all_texts_extracted,True
8,inventory_all_pdf_url_present,True
9,texts_all_pdf_url_present,True


# 9. Descarga física de PDFs RTVE

Una vez obtenidos los enlaces finales a PDF para los 167 documentos, se descargan físicamente los archivos en:

`data/raw/pdfs_rtve/`

La descarga se controla mediante `pdf_manifest_rtve.csv`, que registra para cada documento:

- `doc_id`;
- `pdf_url`;
- ruta local esperada;
- estado de descarga;
- tamaño esperado del archivo;
- tamaño real descargado.

El objetivo de esta sección es conservar una copia local de los documentos originales y dejar trazabilidad entre:

`doc_id` → `texto completo` → `PDF original`.

Los PDFs descargados no se añaden automáticamente a Git. Primero se revisará su tamaño total para decidir si conviene versionarlos o entregarlos aparte en el ZIP final.

In [49]:
# ============================================================
# 9. Descarga física de PDFs RTVE
# ============================================================

from time import sleep

# Si el manifiesto no está cargado en memoria, se lee desde disco.
if "df_pdf_manifest_rtve" not in globals():
    df_pdf_manifest_rtve = pd.read_csv(PDF_MANIFEST_RTVE_PATH)

def download_pdf_file(
    pdf_url: str,
    local_path: Path,
    timeout: int = 60,
    chunk_size: int = 1024 * 1024,
) -> dict:
    """
    Descarga un PDF desde una URL y lo guarda en local.

    Parameters
    ----------
    pdf_url : str
        URL final del PDF.
    local_path : Path
        Ruta local donde se guardará el archivo.
    timeout : int
        Tiempo máximo de espera.
    chunk_size : int
        Tamaño de bloque para descarga en streaming.

    Returns
    -------
    dict
        Resultado de la descarga.
    """
    result = {
        "download_ok": False,
        "file_size_bytes": 0,
        "download_error": None,
    }

    try:
        if pd.isna(pdf_url) or not str(pdf_url).startswith("http"):
            result["download_error"] = "URL de PDF inválida"
            return result

        local_path.parent.mkdir(parents=True, exist_ok=True)

        response = requests.get(
            pdf_url,
            headers=REQUEST_HEADERS,
            timeout=timeout,
            stream=True,
        )
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "").lower()
        if "application/pdf" not in content_type:
            result["download_error"] = f"Content-Type no esperado: {content_type}"
            return result

        with open(local_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)

        file_size = local_path.stat().st_size if local_path.exists() else 0

        result["download_ok"] = file_size > 0
        result["file_size_bytes"] = file_size

    except Exception as exc:
        result["download_error"] = str(exc)

    return result


download_results = []

for idx, row in df_pdf_manifest_rtve.iterrows():
    doc_id = row["doc_id"]
    pdf_url = row["pdf_url"]
    local_pdf_path = PROJECT_ROOT / row["local_pdf_path"]

    if (idx + 1) % 25 == 0 or idx == 0:
        print(f"Descargando PDF {idx + 1}/{len(df_pdf_manifest_rtve)}: {doc_id}")

    result = download_pdf_file(pdf_url=pdf_url, local_path=local_pdf_path)

    download_results.append({
        "doc_id": doc_id,
        "download_ok": result["download_ok"],
        "file_size_bytes": result["file_size_bytes"],
        "download_error": result["download_error"],
    })

    sleep(0.10)

df_pdf_download_results = pd.DataFrame(download_results)

print("Descarga completada.")
print(f"PDFs procesados: {len(df_pdf_download_results)}")
print(f"PDFs descargados correctamente: {df_pdf_download_results['download_ok'].sum()}")
print(f"Errores de descarga: {df_pdf_download_results['download_error'].notna().sum()}")

df_pdf_download_results.head()

Descargando PDF 1/167: rtve_1860
Descargando PDF 25/167: rtve_1836
Descargando PDF 50/167: rtve_1811
Descargando PDF 75/167: rtve_1786
Descargando PDF 100/167: rtve_1761
Descargando PDF 125/167: rtve_1736
Descargando PDF 150/167: rtve_1711
Descarga completada.
PDFs procesados: 167
PDFs descargados correctamente: 167
Errores de descarga: 0


,doc_id,download_ok,file_size_bytes,download_error
0,rtve_1860,True,771201,None
1,rtve_1859,True,979124,None
2,rtve_1858,True,1241724,None
3,rtve_1857,True,1680589,None
4,rtve_1856,True,973226,None


In [50]:
# ------------------------------------------------------------
# 9.2. Actualización del manifiesto de PDFs
# ------------------------------------------------------------

df_pdf_manifest_rtve_updated = (
    df_pdf_manifest_rtve
    .drop(columns=["download_ok", "file_size_bytes"], errors="ignore")
    .merge(df_pdf_download_results, on="doc_id", how="left")
)

# Reordenar columnas
df_pdf_manifest_rtve_updated = df_pdf_manifest_rtve_updated[
    [
        "doc_id",
        "source_document_id",
        "pdf_url",
        "original_url",
        "local_pdf_path",
        "expected_size_bytes",
        "download_ok",
        "file_size_bytes",
        "download_error",
    ]
]

df_pdf_manifest_rtve_updated.to_csv(PDF_MANIFEST_RTVE_PATH, index=False, encoding="utf-8")

df_pdf_manifest_rtve = df_pdf_manifest_rtve_updated.copy()

print(f"Manifiesto actualizado en: {PDF_MANIFEST_RTVE_PATH.relative_to(PROJECT_ROOT)}")
print(f"PDFs descargados correctamente: {df_pdf_manifest_rtve['download_ok'].sum()} / {len(df_pdf_manifest_rtve)}")

df_pdf_manifest_rtve.head()

Manifiesto actualizado en: data/interim/pdf_manifest_rtve.csv
PDFs descargados correctamente: 167 / 167


,doc_id,source_document_id,pdf_url,original_url,local_pdf_path,expected_size_bytes,download_ok,file_size_bytes,download_error
0,rtve_1860,1860,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,https://23fbuscador.rtve.es/document/ocr/1860/original,data/raw/pdfs_rtve/rtve_1860.pdf,771201,True,771201,None
1,rtve_1859,1859,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,https://23fbuscador.rtve.es/document/ocr/1859/original,data/raw/pdfs_rtve/rtve_1859.pdf,979124,True,979124,None
2,rtve_1858,1858,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,https://23fbuscador.rtve.es/document/ocr/1858/original,data/raw/pdfs_rtve/rtve_1858.pdf,1241724,True,1241724,None
3,rtve_1857,1857,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,https://23fbuscador.rtve.es/document/ocr/1857/original,data/raw/pdfs_rtve/rtve_1857.pdf,1680589,True,1680589,None
4,rtve_1856,1856,https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,https://23fbuscador.rtve.es/document/ocr/1856/original,data/raw/pdfs_rtve/rtve_1856.pdf,973226,True,973226,None


In [51]:
# ------------------------------------------------------------
# 9.3. Validación final de PDFs descargados
# ------------------------------------------------------------

pdf_download_validation = {
    "manifest_rows": len(df_pdf_manifest_rtve),
    "expected_rows": len(df_inventory_rtve),
    "matches_inventory_rows": len(df_pdf_manifest_rtve) == len(df_inventory_rtve),

    "unique_doc_id": df_pdf_manifest_rtve["doc_id"].nunique(),
    "all_doc_id_present": df_pdf_manifest_rtve["doc_id"].notna().all(),

    "all_pdf_url_present": df_pdf_manifest_rtve["pdf_url"].notna().all(),
    "all_local_paths_present": df_pdf_manifest_rtve["local_pdf_path"].notna().all(),

    "download_ok_count": int(df_pdf_manifest_rtve["download_ok"].sum()),
    "download_failed_count": int((~df_pdf_manifest_rtve["download_ok"]).sum()),

    "all_downloads_ok": df_pdf_manifest_rtve["download_ok"].all(),
    "all_files_non_empty": (df_pdf_manifest_rtve["file_size_bytes"] > 0).all(),

    "total_size_mb": round(df_pdf_manifest_rtve["file_size_bytes"].sum() / (1024 ** 2), 2),
    "min_file_size_bytes": int(df_pdf_manifest_rtve["file_size_bytes"].min()),
    "median_file_size_bytes": float(df_pdf_manifest_rtve["file_size_bytes"].median()),
    "max_file_size_bytes": int(df_pdf_manifest_rtve["file_size_bytes"].max()),
}

df_pdf_download_validation = pd.DataFrame(
    pdf_download_validation.items(),
    columns=["check", "value"]
)

df_pdf_download_validation

,check,value
0,manifest_rows,167
1,expected_rows,167
2,matches_inventory_rows,True
3,unique_doc_id,167
4,all_doc_id_present,True
5,all_pdf_url_present,True
6,all_local_paths_present,True
7,download_ok_count,167
8,download_failed_count,0
9,all_downloads_ok,True


In [52]:
# ------------------------------------------------------------
# 9.4. Revisión de errores de descarga
# ------------------------------------------------------------

df_failed_pdf_downloads = df_pdf_manifest_rtve[
    ~df_pdf_manifest_rtve["download_ok"]
].copy()

print(f"PDFs con descarga fallida: {len(df_failed_pdf_downloads)}")

if len(df_failed_pdf_downloads) > 0:
    display(
        df_failed_pdf_downloads[
            [
                "doc_id",
                "pdf_url",
                "local_pdf_path",
                "download_error",
            ]
        ].head(20)
    )

PDFs con descarga fallida: 0


### 9.5. Conclusión de la descarga de PDFs

La descarga física de PDFs se ha realizado correctamente a partir de los enlaces obtenidos desde las páginas de detalle de RTVE.

Se han procesado 167 documentos y se han descargado correctamente los 167 PDFs, sin errores de descarga. Todos los archivos descargados tienen tamaño mayor que cero.

Cada PDF se ha guardado con un nombre estable basado en `doc_id`, evitando problemas derivados de títulos largos, caracteres especiales o títulos repetidos.

El archivo `pdf_manifest_rtve.csv` queda como tabla de control de descarga y permite relacionar:

- `doc_id`;
- URL final del PDF;
- ruta local del archivo;
- estado de descarga;
- tamaño esperado;
- tamaño real descargado.

El tamaño total de los PDFs descargados es de aproximadamente 258 MB. Por este motivo, los PDFs no se incorporan directamente al repositorio Git en esta fase; se conservarán localmente y podrán incluirse en el ZIP final de entrega si procede.

Con esto, el bloque RTVE queda formado por tres productos principales:

- `inventory_rtve.csv`: metadatos documentales y enlace final al PDF.
- `document_texts_rtve.csv`: textos completos y métricas básicas.
- `pdf_manifest_rtve.csv`: control de PDFs descargados.

El siguiente paso será revisar La Moncloa como fuente institucional de contraste para comprobar si existe algún documento oficial que no esté cubierto por el corpus RTVE.

## 10. Inventario oficial de La Moncloa

Tras construir el corpus base desde RTVE, se revisa la página oficial de La Moncloa como fuente institucional de contraste.

El objetivo no es duplicar PDFs ni volver a descargar todos los documentos, sino comprobar si la relación oficial de La Moncloa contiene algún documento que no esté ya cubierto por el corpus RTVE.

Para ello se construirá un inventario específico de Moncloa con:

- identificador interno (`moncloa_id`);
- título o texto visible del enlace;
- URL del PDF;
- nombre del archivo PDF;
- versión normalizada del título o nombre del archivo.

Después, este inventario se comparará con `inventory_rtve.csv`.

La comparación no se hará PDF a PDF, sino mediante metadatos y nombres de archivo. Solo si aparece algún documento claramente presente en Moncloa y ausente en RTVE se valorará descargarlo o incorporarlo al corpus.

In [53]:
# ============================================================
# 10. Inventario oficial de La Moncloa
# ============================================================

# ------------------------------------------------------------
# 10.1. Inspección inicial de la página de La Moncloa
# ------------------------------------------------------------

response_moncloa = fetch_response(URL_MONCLOA)
html_moncloa = response_moncloa.text
soup_moncloa = BeautifulSoup(html_moncloa, "html.parser")

print("Página de La Moncloa descargada correctamente.")
print(f"URL: {URL_MONCLOA}")
print(f"Status code: {response_moncloa.status_code}")
print(f"Content-Type: {response_moncloa.headers.get('Content-Type')}")
print(f"Número de caracteres del HTML: {len(html_moncloa):,}")
print(f"Título HTML: {soup_moncloa.title.get_text(strip=True) if soup_moncloa.title else None}")

moncloa_text = soup_moncloa.get_text(" ", strip=True)

search_terms_moncloa = [
    "23-F",
    "documentos desclasificados",
    "Interior",
    "Defensa",
    "Exteriores",
    ".pdf",
    "pdf",
]

print("\nTérminos clave:")
for term in search_terms_moncloa:
    print(f"'{term}': {term.lower() in moncloa_text.lower()}")

Página de La Moncloa descargada correctamente.
URL: https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx
Status code: 200
Content-Type: text/html; charset=utf-8
Número de caracteres del HTML: 149,765
Título HTML: La Moncloa. Documentos desclasificados relativos al 23-F | 25/02/2026 [Consejo de Ministros]

Términos clave:
'23-F': True
'documentos desclasificados': True
'Interior': True
'Defensa': True
'Exteriores': True
'.pdf': False
'pdf': False


In [54]:
# ------------------------------------------------------------
# 10.2. Extracción inicial de enlaces a PDFs en La Moncloa
# ------------------------------------------------------------

moncloa_links = []

for a in soup_moncloa.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = urljoin(URL_MONCLOA, a["href"])

    if ".pdf" in href.lower() or "document" in href.lower() or "23f" in href.lower():
        moncloa_links.append({
            "text": text,
            "href": href,
        })

df_moncloa_links = (
    pd.DataFrame(moncloa_links)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Enlaces candidatos detectados en La Moncloa: {len(df_moncloa_links)}")

df_moncloa_links.head(50)

Enlaces candidatos detectados en La Moncloa: 170


,text,href
0,,https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx?mode=Dark
1,Cerrar,https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx
2,BOE,https://www.boe.es/boe/dias/2026/02/25/pdfs/BOE-A-2026-4351.pdf
3,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
4,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
5,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
6,"Documentación con una presunta planificación del golpe, manuscrita (1980).",https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
7,Documento manuscrito de posible planificación del golpe.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
8,Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
9,Nota del EM de la Guardia Civil con una secuencia parcial de los hechos del asalto al Congreso.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...


In [55]:
# ------------------------------------------------------------
# 10.3. Filtrado de enlaces PDF reales
# ------------------------------------------------------------

df_moncloa_pdf_links = df_moncloa_links[
    df_moncloa_links["href"].str.lower().str.contains(".pdf", regex=False)
].copy()

df_moncloa_pdf_links = df_moncloa_pdf_links.reset_index(drop=True)

print(f"Enlaces PDF detectados en La Moncloa: {len(df_moncloa_pdf_links)}")

df_moncloa_pdf_links.head(50)

Enlaces PDF detectados en La Moncloa: 156


,text,href
0,BOE,https://www.boe.es/boe/dias/2026/02/25/pdfs/BOE-A-2026-4351.pdf
1,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
2,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
3,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
4,"Documentación con una presunta planificación del golpe, manuscrita (1980).",https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
5,Documento manuscrito de posible planificación del golpe.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
6,Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
7,Nota del EM de la Guardia Civil con una secuencia parcial de los hechos del asalto al Congreso.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
8,Télex interiores y de agencias recibidos en 2ª sección EM el día 23-F informando del estado de situación (23 de febr...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...
9,Oficio zona País Vasco que expresa una comunicación del teniente coronel Tejero sobre la posible situación de tensió...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...


### 10.4. Construcción del inventario de La Moncloa

A partir de los enlaces PDF detectados en la página oficial de La Moncloa, se construye un inventario estructurado.

Este inventario no implica descargar los PDFs. Su objetivo es servir como fuente institucional de contraste frente al corpus obtenido desde RTVE.

Se conservarán todos los PDFs detectados, incluido el enlace al BOE, pero se marcará explícitamente si un registro corresponde al BOE para excluirlo de la comparación documental principal.

Columnas principales:

- `moncloa_id`: identificador interno del inventario Moncloa.
- `source`: fuente del registro.
- `title`: texto visible del enlace.
- `pdf_url`: enlace al PDF.
- `pdf_filename`: nombre del archivo PDF.
- `section`: bloque aproximado extraído de la URL, por ejemplo Interior, Defensa o Exteriores.
- `subsection`: subcarpeta institucional si existe, por ejemplo Guardia Civil, Policía o CNI.
- `is_boe`: indica si el enlace corresponde al BOE.
- `include_in_comparison`: indica si el documento debe compararse contra RTVE.

In [56]:
# ------------------------------------------------------------
# 10.4. Construcción del inventario de La Moncloa
# ------------------------------------------------------------

from urllib.parse import unquote

def extract_filename_from_url(url: str) -> str:
    """
    Extrae el nombre de archivo desde una URL.
    """
    if pd.isna(url) or url is None:
        return None

    path = urlparse(url).path
    filename = Path(path).name
    return unquote(filename)


def extract_moncloa_section(url: str) -> str:
    """
    Extrae una sección aproximada desde la URL de La Moncloa.
    """
    if pd.isna(url) or url is None:
        return None

    url_lower = url.lower()

    if "boe.es" in url_lower:
        return "boe"
    if "/interior/" in url_lower:
        return "interior"
    if "/defensa/" in url_lower:
        return "defensa"
    if "/exteriores/" in url_lower:
        return "exteriores"

    return "other"


def extract_moncloa_subsection(url: str) -> str:
    """
    Extrae una subsección aproximada desde la URL de La Moncloa.
    """
    if pd.isna(url) or url is None:
        return None

    url_lower = url.lower()

    known_subsections = [
        "guardia-civil",
        "policia",
        "archivo",
        "cni",
    ]

    for subsection in known_subsections:
        if f"/{subsection}/" in url_lower:
            return subsection

    return None


df_inventory_moncloa = df_moncloa_pdf_links.copy()

df_inventory_moncloa = df_inventory_moncloa.rename(
    columns={
        "text": "title",
        "href": "pdf_url",
    }
)

df_inventory_moncloa["source"] = "moncloa"
df_inventory_moncloa["pdf_filename"] = df_inventory_moncloa["pdf_url"].apply(extract_filename_from_url)
df_inventory_moncloa["section"] = df_inventory_moncloa["pdf_url"].apply(extract_moncloa_section)
df_inventory_moncloa["subsection"] = df_inventory_moncloa["pdf_url"].apply(extract_moncloa_subsection)
df_inventory_moncloa["is_boe"] = df_inventory_moncloa["section"].eq("boe")
df_inventory_moncloa["include_in_comparison"] = ~df_inventory_moncloa["is_boe"]

# Identificador interno estable para Moncloa.
df_inventory_moncloa["moncloa_id"] = [
    f"moncloa_{i:04d}" for i in range(1, len(df_inventory_moncloa) + 1)
]

df_inventory_moncloa = df_inventory_moncloa[
    [
        "moncloa_id",
        "source",
        "title",
        "pdf_url",
        "pdf_filename",
        "section",
        "subsection",
        "is_boe",
        "include_in_comparison",
    ]
]

print(f"Registros PDF en inventario Moncloa: {len(df_inventory_moncloa)}")
print(f"Registros comparables excluyendo BOE: {df_inventory_moncloa['include_in_comparison'].sum()}")

df_inventory_moncloa.head()

Registros PDF en inventario Moncloa: 156
Registros comparables excluyendo BOE: 155


,moncloa_id,source,title,pdf_url,pdf_filename,section,subsection,is_boe,include_in_comparison
0,moncloa_0001,moncloa,BOE,https://www.boe.es/boe/dias/2026/02/25/pdfs/BOE-A-2026-4351.pdf,BOE-A-2026-4351.pdf,boe,None,True,False
1,moncloa_0002,moncloa,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_1._Conversacion_telefonica_GARCIA_CARRES_y_Tcol._TEJERO.pdf,interior,guardia-civil,False,True
2,moncloa_0003,moncloa,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,interior,guardia-civil,False,True
3,moncloa_0004,moncloa,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf,interior,guardia-civil,False,True
4,moncloa_0005,moncloa,"Documentación con una presunta planificación del golpe, manuscrita (1980).",https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_4._Documento_planificacion_del_golpe.pdf,interior,guardia-civil,False,True


In [57]:
# ------------------------------------------------------------
# 10.5. Validación del inventario de La Moncloa
# ------------------------------------------------------------

moncloa_inventory_validation = {
    "n_rows": len(df_inventory_moncloa),
    "n_unique_moncloa_id": df_inventory_moncloa["moncloa_id"].nunique(),
    "moncloa_id_all_present": df_inventory_moncloa["moncloa_id"].notna().all(),

    "n_missing_title": df_inventory_moncloa["title"].isna().sum(),
    "n_missing_pdf_url": df_inventory_moncloa["pdf_url"].isna().sum(),
    "n_missing_pdf_filename": df_inventory_moncloa["pdf_filename"].isna().sum(),

    "n_boe": int(df_inventory_moncloa["is_boe"].sum()),
    "n_include_in_comparison": int(df_inventory_moncloa["include_in_comparison"].sum()),

    "n_unique_pdf_url": df_inventory_moncloa["pdf_url"].nunique(),
    "pdf_url_all_unique": df_inventory_moncloa["pdf_url"].nunique() == len(df_inventory_moncloa),

    "n_unique_pdf_filename": df_inventory_moncloa["pdf_filename"].nunique(),
    "pdf_filename_all_unique": df_inventory_moncloa["pdf_filename"].nunique() == len(df_inventory_moncloa),
}

df_moncloa_inventory_validation = pd.DataFrame(
    moncloa_inventory_validation.items(),
    columns=["check", "value"]
)

df_moncloa_inventory_validation

,check,value
0,n_rows,156
1,n_unique_moncloa_id,156
2,moncloa_id_all_present,True
3,n_missing_title,0
4,n_missing_pdf_url,0
5,n_missing_pdf_filename,0
6,n_boe,1
7,n_include_in_comparison,155
8,n_unique_pdf_url,156
9,pdf_url_all_unique,True


In [58]:
# ------------------------------------------------------------
# 10.6. Distribución de documentos Moncloa por sección
# ------------------------------------------------------------

df_moncloa_section_counts = (
    df_inventory_moncloa
    .groupby(["section", "subsection"], dropna=False)
    .size()
    .reset_index(name="n_documents")
    .sort_values(["section", "subsection"])
)

df_moncloa_section_counts

,section,subsection,n_documents
0,boe,NaN,1
1,defensa,cni,84
2,defensa,NaN,24
3,exteriores,NaN,19
4,interior,archivo,7
5,interior,guardia-civil,11
6,interior,policia,10


In [59]:
# ------------------------------------------------------------
# 10.7. Guardado del inventario de La Moncloa
# ------------------------------------------------------------

can_save_moncloa_inventory = (
    len(df_inventory_moncloa) > 0
    and df_inventory_moncloa["moncloa_id"].notna().all()
    and df_inventory_moncloa["moncloa_id"].nunique() == len(df_inventory_moncloa)
    and df_inventory_moncloa["pdf_url"].notna().all()
)

if can_save_moncloa_inventory:
    df_inventory_moncloa.to_csv(INVENTORY_MONCLOA_PATH, index=False, encoding="utf-8")
    print(f"Inventario Moncloa guardado en: {INVENTORY_MONCLOA_PATH.relative_to(PROJECT_ROOT)}")
else:
    print("No se guarda el inventario Moncloa porque no pasa las validaciones mínimas.")

Inventario Moncloa guardado en: data/interim/inventory_moncloa.csv


## 11. Comparación entre RTVE y La Moncloa

Una vez construido el inventario de La Moncloa, se compara con el inventario de RTVE.

El objetivo no es comparar PDF a PDF, sino comprobar si la cobertura documental obtenida desde RTVE parece cubrir los documentos listados oficialmente por La Moncloa.

La comparación se hace mediante metadatos:

- título de RTVE;
- título visible en La Moncloa;
- nombre del archivo PDF de Moncloa;
- nombre del archivo PDF de RTVE;
- similitud textual entre cadenas normalizadas.

La diferencia inicial de conteo no implica necesariamente que falten documentos. RTVE contiene 167 registros, mientras que La Moncloa contiene 155 documentos comparables si se excluye el BOE. Esta diferencia puede deberse a criterios distintos de división o agrupación documental.

El resultado se guardará en:

`data/interim/source_comparison.csv`

In [60]:
# ============================================================
# 11. Comparación entre RTVE y La Moncloa
# ============================================================

from difflib import SequenceMatcher
import unicodedata

def normalize_text_for_matching(text: str) -> str:
    """
    Normaliza texto para comparación aproximada entre fuentes.
    """
    if pd.isna(text) or text is None:
        return ""

    text = str(text).lower().strip()

    # Quitar acentos
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    # Quitar extensión PDF y separadores frecuentes
    text = text.replace(".pdf", " ")
    text = text.replace("_", " ")
    text = text.replace("-", " ")

    # Mantener solo letras, números y espacios
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalizar espacios
    text = re.sub(r"\s+", " ", text).strip()

    return text


def similarity_score(a: str, b: str) -> float:
    """
    Calcula similitud entre dos cadenas normalizadas.
    """
    a_norm = normalize_text_for_matching(a)
    b_norm = normalize_text_for_matching(b)

    if not a_norm or not b_norm:
        return 0.0

    return SequenceMatcher(None, a_norm, b_norm).ratio()

In [61]:
# ------------------------------------------------------------
# 11.1. Preparación de inventarios para comparación
# ------------------------------------------------------------

# Moncloa: solo documentos comparables, excluyendo BOE.
df_moncloa_compare = df_inventory_moncloa[
    df_inventory_moncloa["include_in_comparison"]
].copy()

# RTVE: inventario ya construido.
df_rtve_compare = df_inventory_rtve.copy()

# Extraer nombre de PDF RTVE desde pdf_url.
df_rtve_compare["pdf_filename"] = df_rtve_compare["pdf_url"].apply(extract_filename_from_url)

# Campos normalizados.
df_moncloa_compare["title_norm"] = df_moncloa_compare["title"].apply(normalize_text_for_matching)
df_moncloa_compare["pdf_filename_norm"] = df_moncloa_compare["pdf_filename"].apply(normalize_text_for_matching)

df_rtve_compare["title_norm"] = df_rtve_compare["title"].apply(normalize_text_for_matching)
df_rtve_compare["pdf_filename_norm"] = df_rtve_compare["pdf_filename"].apply(normalize_text_for_matching)

print(f"Documentos RTVE: {len(df_rtve_compare)}")
print(f"Documentos Moncloa comparables: {len(df_moncloa_compare)}")

df_moncloa_compare.head()

Documentos RTVE: 167
Documentos Moncloa comparables: 155


,moncloa_id,source,title,pdf_url,pdf_filename,section,subsection,is_boe,include_in_comparison,title_norm,pdf_filename_norm
1,moncloa_0002,moncloa,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_1._Conversacion_telefonica_GARCIA_CARRES_y_Tcol._TEJERO.pdf,interior,guardia-civil,False,True,transcripcion de conversacion telefonica de presuntamente garcia carres y tejero mientras posiblemente el segundo se...,23f 1 conversacion telefonica garcia carres y tcol tejero
2,moncloa_0003,moncloa,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,interior,guardia-civil,False,True,transcripcion de conversacion telefonica de garcia carres con otra persona despues de la citada,23f 2 conversacion telefonica garcia carres
3,moncloa_0004,moncloa,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf,interior,guardia-civil,False,True,conversaciones telefonicas de presuntamente la unidad militar el pardo 24 de febrero del 1981,23f 3 conversaciones telefonicas unidad militar el pardo
4,moncloa_0005,moncloa,"Documentación con una presunta planificación del golpe, manuscrita (1980).",https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_4._Documento_planificacion_del_golpe.pdf,interior,guardia-civil,False,True,documentacion con una presunta planificacion del golpe manuscrita 1980,23f 4 documento planificacion del golpe
5,moncloa_0006,moncloa,Documento manuscrito de posible planificación del golpe.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_5._Documento_manuscrito_planificacion_del_golpe.pdf,interior,guardia-civil,False,True,documento manuscrito de posible planificacion del golpe,23f 5 documento manuscrito planificacion del golpe


In [62]:
# ------------------------------------------------------------
# 11.2. Búsqueda del mejor candidato RTVE para cada documento Moncloa
# ------------------------------------------------------------

comparison_rows = []

for _, moncloa_row in df_moncloa_compare.iterrows():
    best_match = None
    best_score = -1
    best_method = None

    for _, rtve_row in df_rtve_compare.iterrows():
        # Comparación por título
        title_score = similarity_score(moncloa_row["title"], rtve_row["title"])

        # Comparación por nombre de PDF
        filename_score = similarity_score(moncloa_row["pdf_filename"], rtve_row["pdf_filename"])

        # Comparación combinada: damos más peso al título visible
        combined_score = (0.7 * title_score) + (0.3 * filename_score)

        if combined_score > best_score:
            best_score = combined_score
            best_match = rtve_row
            best_method = "title_pdf_filename_similarity"

    if best_score >= 0.70:
        match_status = "high_confidence_match"
    elif best_score >= 0.45:
        match_status = "possible_match"
    else:
        match_status = "no_clear_match"

    comparison_rows.append({
        "moncloa_id": moncloa_row["moncloa_id"],
        "moncloa_title": moncloa_row["title"],
        "moncloa_pdf_url": moncloa_row["pdf_url"],
        "moncloa_pdf_filename": moncloa_row["pdf_filename"],
        "moncloa_section": moncloa_row["section"],
        "moncloa_subsection": moncloa_row["subsection"],

        "matched_rtve_doc_id": best_match["doc_id"] if best_match is not None else None,
        "rtve_title": best_match["title"] if best_match is not None else None,
        "rtve_pdf_url": best_match["pdf_url"] if best_match is not None else None,
        "rtve_pdf_filename": best_match["pdf_filename"] if best_match is not None else None,

        "match_method": best_method,
        "match_score": round(best_score, 4),
        "match_status": match_status,
        "notes": None,
    })

df_source_comparison = pd.DataFrame(comparison_rows)

print("Distribución de estados de matching:")
display(df_source_comparison["match_status"].value_counts(dropna=False))

df_source_comparison.head()

Distribución de estados de matching:


match_status
high_confidence_match    150
possible_match             3
no_clear_match             2
Name: count, dtype: int64

,moncloa_id,moncloa_title,moncloa_pdf_url,moncloa_pdf_filename,moncloa_section,moncloa_subsection,matched_rtve_doc_id,rtve_title,rtve_pdf_url,rtve_pdf_filename,match_method,match_score,match_status,notes
0,moncloa_0002,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_1._Conversacion_telefonica_GARCIA_CARRES_y_Tcol._TEJERO.pdf,interior,guardia-civil,rtve_1694,Transcripción de conversación telefónica de (presuntamente) García Carres y Tejero mientras (posiblemente) el segund...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/02_2026_transcripcion_de_conversacion_telefonica_de_pre...,02_2026_transcripcion_de_conversacion_telefonica_de_presuntamente_garcia_carres_y_tejero.pdf,title_pdf_filename_similarity,0.9028,high_confidence_match,None
1,moncloa_0003,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,interior,guardia-civil,rtve_1695,Transcripción de conversación telefónica de García Carres con otra persona después de la citada.,https://www.rtve.es/contenidos/documentos/23f-desclasificado/03_2026_transcripcion_de_conversacion_telefonica_de_gar...,03_2026_transcripcion_de_conversacion_telefonica_de_garcia_carres_con_otra_persona_despu.pdf,title_pdf_filename_similarity,0.8878,high_confidence_match,None
2,moncloa_0004,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_3._Conversaciones_telefonicas_unidad_militar_El_Pardo.pdf,interior,guardia-civil,rtve_1696,Conversaciones telefónicas de (presuntamente) la unidad militar El Pardo (24 de febrero del 1981).,https://www.rtve.es/contenidos/documentos/23f-desclasificado/04_1981_conversaciones_telefonicas_de_presuntamente_la_...,04_1981_conversaciones_telefonicas_de_presuntamente_la_unidad_militar_el_pardo_24_de_feb.pdf,title_pdf_filename_similarity,0.9167,high_confidence_match,None
3,moncloa_0005,"Documentación con una presunta planificación del golpe, manuscrita (1980).",https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_4._Documento_planificacion_del_golpe.pdf,interior,guardia-civil,rtve_1697,"""Documentación con una presunta planificación del golpe",https://www.rtve.es/contenidos/documentos/23f-desclasificado/05_1980_documentacion_con_una_presunta_planificacion_de...,05_1980_documentacion_con_una_presunta_planificacion_del_golpe_manuscrita_1980.pdf,title_pdf_filename_similarity,0.7892,high_confidence_match,None
4,moncloa_0006,Documento manuscrito de posible planificación del golpe.,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/guardia-civi...,23F_5._Documento_manuscrito_planificacion_del_golpe.pdf,interior,guardia-civil,rtve_1698,Documento manuscrito de posible planificación del golpe.,https://www.rtve.es/contenidos/documentos/23f-desclasificado/06_2026_documento_manuscrito_de_posible_planificacion_d...,06_2026_documento_manuscrito_de_posible_planificacion_del_golpe.pdf,title_pdf_filename_similarity,0.9442,high_confidence_match,None


In [63]:
# ------------------------------------------------------------
# 11.3. Revisión de documentos Moncloa sin match claro
# ------------------------------------------------------------

df_no_clear_matches = df_source_comparison[
    df_source_comparison["match_status"] == "no_clear_match"
].copy()

df_possible_matches = df_source_comparison[
    df_source_comparison["match_status"] == "possible_match"
].copy()

print(f"No clear matches: {len(df_no_clear_matches)}")
print(f"Possible matches: {len(df_possible_matches)}")

df_no_clear_matches[
    [
        "moncloa_id",
        "moncloa_title",
        "moncloa_pdf_filename",
        "matched_rtve_doc_id",
        "rtve_title",
        "rtve_pdf_filename",
        "match_score",
    ]
].head(30)

No clear matches: 2
Possible matches: 3


,moncloa_id,moncloa_title,moncloa_pdf_filename,matched_rtve_doc_id,rtve_title,rtve_pdf_filename,match_score
23,moncloa_0025,"Juicio del 23-F: acotaciones al desarrollo del juicio, notas procesales sobre el consejo de guerra, nota sobre posib...",3_Juicio_del_23-F_desp.pdf,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesales_sobre_el_c.pdf,0.3691
27,moncloa_0029,"Notas de 1983 sobre ""Libertad de condenados por el 23-F"", ""Sobre petición de indulto"", ""Vista del recurso de casació...",7_Notas_1983_desp.pdf,rtve_1790,"""Notas de 1983 sobre """"Libertad de condenados por el 23-F""""",29_1983_notas_de_1983_sobre_libertad_de_condenados_por_el_23_f_sobre_peticion_de_indulto.pdf,0.4216


In [64]:
# ------------------------------------------------------------
# 11.4. Guardado de la comparación entre fuentes
# ------------------------------------------------------------

df_source_comparison.to_csv(SOURCE_COMPARISON_PATH, index=False, encoding="utf-8")

print(f"Comparación RTVE vs Moncloa guardada en: {SOURCE_COMPARISON_PATH.relative_to(PROJECT_ROOT)}")
print(f"Filas guardadas: {len(df_source_comparison)}")

Comparación RTVE vs Moncloa guardada en: data/interim/source_comparison.csv
Filas guardadas: 155


In [65]:
# ------------------------------------------------------------
# 11.5. Revisión de matches dudosos
# ------------------------------------------------------------

df_matches_to_review = df_source_comparison[
    df_source_comparison["match_status"].isin(["possible_match", "no_clear_match"])
].copy()

df_matches_to_review[
    [
        "moncloa_id",
        "moncloa_title",
        "moncloa_pdf_filename",
        "matched_rtve_doc_id",
        "rtve_title",
        "rtve_pdf_filename",
        "match_score",
        "match_status",
    ]
]

,moncloa_id,moncloa_title,moncloa_pdf_filename,matched_rtve_doc_id,rtve_title,rtve_pdf_filename,match_score,match_status
23,moncloa_0025,"Juicio del 23-F: acotaciones al desarrollo del juicio, notas procesales sobre el consejo de guerra, nota sobre posib...",3_Juicio_del_23-F_desp.pdf,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesales_sobre_el_c.pdf,0.3691,no_clear_match
27,moncloa_0029,"Notas de 1983 sobre ""Libertad de condenados por el 23-F"", ""Sobre petición de indulto"", ""Vista del recurso de casació...",7_Notas_1983_desp.pdf,rtve_1790,"""Notas de 1983 sobre """"Libertad de condenados por el 23-F""""",29_1983_notas_de_1983_sobre_libertad_de_condenados_por_el_23_f_sobre_peticion_de_indulto.pdf,0.4216,no_clear_match
28,moncloa_0030,"Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno, el ministro de Defensa y la ...",Documento_1_R.pdf,rtve_1791,"""Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno",30_1981_guion_que_sirvio_de_base_para_la_reunion_de_sm_el_rey_con_el_presidente_del_gobi.pdf,0.5119,possible_match
46,moncloa_0048,"Estado de opinión sobre las sentencias (sección de contrainformación del EME, sección de contrainteligencia) (7 de j...",Documento_19_R.pdf,rtve_1809,"""Estado de opinión sobre las sentencias (sección de contrainformación del EME",48_1982_estado_de_opinion_sobre_las_sentencias_seccion_de_contrainformacion_del_eme_secc.pdf,0.5806,possible_match
106,moncloa_0108,"Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle, un cabo 1º, dos guardias y ...",Documento_79_R.pdf,rtve_1710,"""Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle",108_1982_interesando_comparecencia_del_capitan_de_la_guardia_civil_jose_ramon_toston_de_l.pdf,0.5267,possible_match


In [66]:
# ------------------------------------------------------------
# 11.6. Documentos RTVE no emparejados con Moncloa
# ------------------------------------------------------------

matched_rtve_ids = set(
    df_source_comparison["matched_rtve_doc_id"]
    .dropna()
    .unique()
)

df_rtve_not_matched_by_moncloa = df_inventory_rtve[
    ~df_inventory_rtve["doc_id"].isin(matched_rtve_ids)
].copy()

print(f"Documentos RTVE no emparejados desde Moncloa: {len(df_rtve_not_matched_by_moncloa)}")

df_rtve_not_matched_by_moncloa[
    [
        "doc_id",
        "source_document_id",
        "title",
        "pages",
        "keywords",
        "pdf_url",
    ]
].head(30)

Documentos RTVE no emparejados desde Moncloa: 26


,doc_id,source_document_id,title,pages,keywords,pdf_url
8,rtve_1852,1852,Vista oral 2/81 del Consejo Supremo de Justicia Militar (5 de marzo de 1982).,8,C/SG/3644/05-03-82 Vista oral Causa 2/81,https://www.rtve.es/contenidos/documentos/23f-desclasificado/91_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
9,rtve_1851,1851,Vista oral 2/81 del Consejo Supremo de Justicia Militar (8 de marzo de 1982).,5,C/SG/3835/08-03-82 NOTA INFORMATIVA VISTA ORAL,https://www.rtve.es/contenidos/documentos/23f-desclasificado/90_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
19,rtve_1841,1841,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de marzo de 1982).,5,N/Ref. C/SG/4.819/24.03.82 VISTA ORAL 2/81 CONSEJO SUPREMO DE JUSTICIA MILITAR,https://www.rtve.es/contenidos/documentos/23f-desclasificado/80_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
28,rtve_1832,1832,Vista oral 2/81 del Consejo Supremo de Justicia Militar (6 de abril de 1982).,6,DTOR C/SG/5.622/06.04.82 VISTA ORAL 2/81,https://www.rtve.es/contenidos/documentos/23f-desclasificado/71_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
29,rtve_1831,1831,Vista oral 2/81 del Consejo Supremo de Justicia Militar (7 de abril de 1982).,6,DTOR N.Ref: C/SG/ 5688/ 07-04-82 VISTA ORAL 2/81,https://www.rtve.es/contenidos/documentos/23f-desclasificado/70_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
46,rtve_1814,1814,Vista oral 2/81 del Consejo Supremo de Justicia Militar (12 de mayo de 1982).,5,DTOR 7497/12-05-82 NOTA INFORMATIVA,https://www.rtve.es/contenidos/documentos/23f-desclasificado/53_1982_vista_oral_281_del_consejo_supremo_de_justicia_...
99,rtve_1761,1761,D.17._AGMAE_R40201_Exp._215,1,S. R. CÂMARA MUNICIPAL DE CAMPO MAIOR SECRETARIA,https://www.rtve.es/contenidos/documentos/23f-desclasificado/154_2026_d17_agmae_r40201_exp_215.pdf
100,rtve_1760,1760,D.16._AGMAE_R40201_Exp._215,1,ASSEMBLEIA MUNICIPAL DO CONCELHO DE CAMPO MAIOR Chanceler do Consulado de Espanha em ELVAS 24. FEB. 1961,https://www.rtve.es/contenidos/documentos/23f-desclasificado/153_2026_d16_agmae_r40201_exp_215.pdf
103,rtve_1757,1757,D.13._AGMAE_R39017_Exp._4,1,Madrid 16 de marzo de 1981 Excmo. Sr. Terence A. Todman,https://www.rtve.es/contenidos/documentos/23f-desclasificado/150_2026_d13_agmae_r39017_exp_4.pdf
105,rtve_1755,1755,D.12._AGMAE_R39017_Exp._4,1,MINISTERIO DE ASUNTOS EXTERIORES Núm. 179 WASHINGTON,https://www.rtve.es/contenidos/documentos/23f-desclasificado/149_2026_d12_agmae_r39017_exp_4.pdf


In [67]:
# ------------------------------------------------------------
# 11.7. Revisión de colisiones: varios documentos Moncloa asignados al mismo RTVE
# ------------------------------------------------------------

df_match_collisions = (
    df_source_comparison
    .groupby("matched_rtve_doc_id", dropna=False)
    .size()
    .reset_index(name="n_moncloa_matches")
    .sort_values("n_moncloa_matches", ascending=False)
)

df_match_collisions = df_match_collisions[
    df_match_collisions["n_moncloa_matches"] > 1
].copy()

print(f"RTVE doc_id con más de un documento Moncloa asignado: {len(df_match_collisions)}")

df_match_collisions.head(30)

RTVE doc_id con más de un documento Moncloa asignado: 9


,matched_rtve_doc_id,n_moncloa_matches
30,rtve_1731,6
126,rtve_1844,3
28,rtve_1724,2
116,rtve_1833,2
125,rtve_1843,2
26,rtve_1721,2
113,rtve_1828,2
31,rtve_1733,2
97,rtve_1811,2


In [68]:
# ------------------------------------------------------------
# 11.8. Detalle de colisiones de matching
# ------------------------------------------------------------

collision_rtve_ids = df_match_collisions["matched_rtve_doc_id"].dropna().tolist()

df_collision_details = df_source_comparison[
    df_source_comparison["matched_rtve_doc_id"].isin(collision_rtve_ids)
].copy()

df_collision_details = df_collision_details.sort_values(
    ["matched_rtve_doc_id", "match_score"],
    ascending=[True, False]
)

df_collision_details[
    [
        "matched_rtve_doc_id",
        "rtve_title",
        "moncloa_id",
        "moncloa_title",
        "moncloa_pdf_filename",
        "match_score",
        "match_status",
    ]
].head(80)

,matched_rtve_doc_id,rtve_title,moncloa_id,moncloa_title,moncloa_pdf_filename,match_score,match_status
115,rtve_1721,SECRETO: oficio dando cuenta toma de declaración.,moncloa_0117,SECRETO: oficio dando cuenta toma de declaración.,Carpeta_21800_Secreto_oficio_dando_cuenta_toma_de_declaracion_1.pdf,0.9571,high_confidence_match
116,rtve_1721,SECRETO: oficio dando cuenta toma de declaración.,moncloa_0118,SECRETO: oficio dando cuenta toma de declaración.,Carpeta_21800_Secreto_oficio_dando_cuenta_toma_de_declaracion_2.pdf,0.9571,high_confidence_match
117,rtve_1724,RESERVADO: oficio dando cuenta toma de declaración.,moncloa_0119,RESERVADO: oficio dando cuenta toma de declaración.,Carpeta_21800_Reservado_oficio_dando_cuenta_toma_de_declaracion_3.pdf,0.9585,high_confidence_match
118,rtve_1724,RESERVADO: oficio dando cuenta toma de declaración.,moncloa_0120,RESERVADO: oficio dando cuenta toma de declaración.,Carpeta_21800_Reservado_oficio_dando_cuenta_toma_de_declaracion_4.pdf,0.9585,high_confidence_match
120,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0122,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_1.pdf,0.9500,high_confidence_match
121,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0123,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_2.pdf,0.9500,high_confidence_match
122,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0124,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_3.pdf,0.9500,high_confidence_match
123,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0125,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_4.pdf,0.9500,high_confidence_match
124,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0126,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_5.pdf,0.9500,high_confidence_match
125,rtve_1731,RESERVADO: comunicación procesamiento implicado.,moncloa_0127,RESERVADO: comunicación procesamiento implicado.,Carpeta_21800_Reservado_comunicacion_procesamiento_de_implicado_6.pdf,0.9500,high_confidence_match


### 11.9. Mejora del matching: fechas y asignación uno-a-uno

La primera comparación detecta un problema: varios documentos de La Moncloa se asignan al mismo documento de RTVE.

Esto ocurre especialmente en títulos muy repetidos, como:

- `Vista oral 2/81 del Consejo Supremo de Justicia Militar (...)`
- `RESERVADO: comunicación procesamiento implicado`
- `SECRETO: oficio dando cuenta toma de declaración`

Por tanto, se ajusta la lógica de comparación:

1. Se extrae la fecha del título cuando aparece entre paréntesis.
2. Si ambos documentos tienen fecha y no coincide, se penaliza el match.
3. Se fuerza una asignación uno-a-uno: cada documento RTVE solo puede ser usado una vez.
4. Los casos dudosos seguirán quedando marcados para revisión manual.

Esta mejora evita que documentos con títulos genéricos o muy parecidos se emparejen incorrectamente.

In [69]:
# ------------------------------------------------------------
# 11.9. Extracción de fechas desde títulos
# ------------------------------------------------------------

SPANISH_MONTHS = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "setiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12",
}

def extract_spanish_date(text: str) -> str | None:
    """
    Extrae fechas del tipo '20 de febrero de 1982' y las normaliza como YYYY-MM-DD.
    """
    if pd.isna(text) or text is None:
        return None

    text_norm = normalize_text_for_matching(text)

    pattern = r"(\d{1,2}) de ([a-z]+) de (\d{4})"
    match = re.search(pattern, text_norm)

    if not match:
        return None

    day, month_name, year = match.groups()
    month = SPANISH_MONTHS.get(month_name)

    if month is None:
        return None

    return f"{year}-{month}-{int(day):02d}"


# Añadir fechas extraídas
df_moncloa_compare["date_extracted"] = df_moncloa_compare["title"].apply(extract_spanish_date)
df_rtve_compare["date_extracted"] = df_rtve_compare["title"].apply(extract_spanish_date)

print("Fechas extraídas en Moncloa:")
print(df_moncloa_compare["date_extracted"].notna().sum(), "/", len(df_moncloa_compare))

print("Fechas extraídas en RTVE:")
print(df_rtve_compare["date_extracted"].notna().sum(), "/", len(df_rtve_compare))

Fechas extraídas en Moncloa:
87 / 155
Fechas extraídas en RTVE:
83 / 167


In [70]:
# ------------------------------------------------------------
# 11.10. Matching mejorado con fecha y asignación uno-a-uno
# ------------------------------------------------------------

def improved_match_score(moncloa_row: pd.Series, rtve_row: pd.Series) -> float:
    """
    Calcula una puntuación de matching entre un documento Moncloa y uno RTVE.

    Criterios:
    - similitud de título;
    - similitud de nombre de PDF;
    - bonus si la fecha coincide;
    - penalización si ambos tienen fecha pero no coincide.
    """
    title_score = similarity_score(moncloa_row["title"], rtve_row["title"])
    filename_score = similarity_score(moncloa_row["pdf_filename"], rtve_row["pdf_filename"])

    score = (0.75 * title_score) + (0.25 * filename_score)

    moncloa_date = moncloa_row.get("date_extracted")
    rtve_date = rtve_row.get("date_extracted")

    if pd.notna(moncloa_date) and pd.notna(rtve_date):
        if moncloa_date == rtve_date:
            score += 0.20
        else:
            score -= 0.35

    return max(0.0, min(score, 1.0))


candidate_rows = []

for _, moncloa_row in df_moncloa_compare.iterrows():
    for _, rtve_row in df_rtve_compare.iterrows():
        score = improved_match_score(moncloa_row, rtve_row)

        candidate_rows.append({
            "moncloa_id": moncloa_row["moncloa_id"],
            "moncloa_title": moncloa_row["title"],
            "moncloa_pdf_url": moncloa_row["pdf_url"],
            "moncloa_pdf_filename": moncloa_row["pdf_filename"],
            "moncloa_section": moncloa_row["section"],
            "moncloa_subsection": moncloa_row["subsection"],
            "moncloa_date": moncloa_row["date_extracted"],

            "matched_rtve_doc_id": rtve_row["doc_id"],
            "rtve_title": rtve_row["title"],
            "rtve_pdf_url": rtve_row["pdf_url"],
            "rtve_pdf_filename": rtve_row["pdf_filename"],
            "rtve_date": rtve_row["date_extracted"],

            "match_score": round(score, 4),
        })

df_match_candidates = pd.DataFrame(candidate_rows)

# Greedy one-to-one: se ordenan todos los candidatos por score descendente.
df_match_candidates = df_match_candidates.sort_values(
    "match_score",
    ascending=False
).reset_index(drop=True)

assigned_moncloa = set()
assigned_rtve = set()
final_matches = []

for _, row in df_match_candidates.iterrows():
    moncloa_id = row["moncloa_id"]
    rtve_id = row["matched_rtve_doc_id"]

    if moncloa_id in assigned_moncloa:
        continue

    if rtve_id in assigned_rtve:
        continue

    assigned_moncloa.add(moncloa_id)
    assigned_rtve.add(rtve_id)

    final_matches.append(row.to_dict())

df_source_comparison_v2 = pd.DataFrame(final_matches)

# Clasificación final por score
def classify_match(score: float) -> str:
    if score >= 0.75:
        return "high_confidence_match"
    elif score >= 0.50:
        return "possible_match"
    else:
        return "no_clear_match"

df_source_comparison_v2["match_status"] = df_source_comparison_v2["match_score"].apply(classify_match)
df_source_comparison_v2["match_method"] = "date_aware_one_to_one_similarity"
df_source_comparison_v2["notes"] = None

print("Distribución de estados de matching mejorado:")
display(df_source_comparison_v2["match_status"].value_counts(dropna=False))

print(f"Moncloa asignados: {df_source_comparison_v2['moncloa_id'].nunique()} / {len(df_moncloa_compare)}")
print(f"RTVE usados: {df_source_comparison_v2['matched_rtve_doc_id'].nunique()} / {len(df_rtve_compare)}")

df_source_comparison_v2.head()

Distribución de estados de matching mejorado:


match_status
high_confidence_match    150
possible_match             3
no_clear_match             2
Name: count, dtype: int64

Moncloa asignados: 155 / 155
RTVE usados: 155 / 167


,moncloa_id,moncloa_title,moncloa_pdf_url,moncloa_pdf_filename,moncloa_section,moncloa_subsection,moncloa_date,matched_rtve_doc_id,rtve_title,rtve_pdf_url,rtve_pdf_filename,rtve_date,match_score,match_status,match_method,notes
0,moncloa_0093,Información integrada (16 de marzo de 1982).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/defensa/cni/Documento...,Documento_64_R.pdf,defensa,cni,1982-03-16,rtve_1854,Información integrada (16 de marzo de 1982).,https://www.rtve.es/contenidos/documentos/23f-desclasificado/93_1982_informacion_integrada_16_de_marzo_de_1982.pdf,93_1982_informacion_integrada_16_de_marzo_de_1982.pdf,1982-03-16,1.0,high_confidence_match,date_aware_one_to_one_similarity,None
1,moncloa_0040,Ambiente en los cuarteles (25 de febrero de 1982).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/defensa/cni/Documento...,Documento_11_R.pdf,defensa,cni,1982-02-25,rtve_1801,Ambiente en los cuarteles (25 de febrero de 1982).,https://www.rtve.es/contenidos/documentos/23f-desclasificado/40_1982_ambiente_en_los_cuarteles_25_de_febrero_de_1982...,40_1982_ambiente_en_los_cuarteles_25_de_febrero_de_1982.pdf,1982-02-25,1.0,high_confidence_match,date_aware_one_to_one_similarity,None
2,moncloa_0023,Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/archivo/1_PN...,1_PN_Informe_Situacion_12-11-81_desp.pdf,interior,archivo,1981-11-12,rtve_1784,Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).,https://www.rtve.es/contenidos/documentos/23f-desclasificado/23_1981_policia_nacional_informe_de_situacion_marca_res...,23_1981_policia_nacional_informe_de_situacion_marca_reservado_confidencial_12_de_noviemb.pdf,1981-11-12,1.0,high_confidence_match,date_aware_one_to_one_similarity,None
3,moncloa_0076,Información integrada (5 de abril de 1982).,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/defensa/cni/Documento...,Documento_47_R.pdf,defensa,cni,1982-04-05,rtve_1837,Información integrada (5 de abril de 1982).,https://www.rtve.es/contenidos/documentos/23f-desclasificado/76_1982_informacion_integrada_5_de_abril_de_1982.pdf,76_1982_informacion_integrada_5_de_abril_de_1982.pdf,1982-04-05,1.0,high_confidence_match,date_aware_one_to_one_similarity,None
4,moncloa_0017,Nota de la Brigada de Información Interior: Ayudas a los implicados en el 23-F. Desde altas instancias castrenses ca...,https://www.lamoncloa.gob.es/consejodeministros/Documents/2026/desclasificacion-documentos-23F/interior/policia/18-0...,18-03-81_NOTA_INFORMATIVA_SOBRE_LA_AYUDA_A_LOS_IMPLICADOS_23F.pdf,interior,policia,1981-03-18,rtve_1778,Nota de la Brigada de Información Interior: Ayudas a los implicados en el 23-F. Desde altas instancias castrenses ca...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/17_1981_nota_de_la_brigada_de_informacion_interior_ayud...,17_1981_nota_de_la_brigada_de_informacion_interior_ayudas_a_los_implicados_en_el_23_f_de.pdf,1981-03-18,1.0,high_confidence_match,date_aware_one_to_one_similarity,None


In [71]:
# ------------------------------------------------------------
# 11.11. Revisión de matches dudosos tras mejora
# ------------------------------------------------------------

df_matches_to_review_v2 = df_source_comparison_v2[
    df_source_comparison_v2["match_status"].isin(["possible_match", "no_clear_match"])
].copy()

print(f"Matches dudosos tras mejora: {len(df_matches_to_review_v2)}")

df_matches_to_review_v2[
    [
        "moncloa_id",
        "moncloa_title",
        "moncloa_pdf_filename",
        "moncloa_date",
        "matched_rtve_doc_id",
        "rtve_title",
        "rtve_pdf_filename",
        "rtve_date",
        "match_score",
        "match_status",
    ]
].head(40)

Matches dudosos tras mejora: 5


,moncloa_id,moncloa_title,moncloa_pdf_filename,moncloa_date,matched_rtve_doc_id,rtve_title,rtve_pdf_filename,rtve_date,match_score,match_status
150,moncloa_0048,"Estado de opinión sobre las sentencias (sección de contrainformación del EME, sección de contrainteligencia) (7 de j...",Documento_19_R.pdf,1982-06-07,rtve_1809,"""Estado de opinión sobre las sentencias (sección de contrainformación del EME",48_1982_estado_de_opinion_sobre_las_sentencias_seccion_de_contrainformacion_del_eme_secc.pdf,None,0.6094,possible_match
151,moncloa_0108,"Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle, un cabo 1º, dos guardias y ...",Documento_79_R.pdf,1982-01-05,rtve_1710,"""Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle",108_1982_interesando_comparecencia_del_capitan_de_la_guardia_civil_jose_ramon_toston_de_l.pdf,None,0.5505,possible_match
152,moncloa_0030,"Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno, el ministro de Defensa y la ...",Documento_1_R.pdf,1981-12-14,rtve_1791,"""Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno",30_1981_guion_que_sirvio_de_base_para_la_reunion_de_sm_el_rey_con_el_presidente_del_gobi.pdf,None,0.5386,possible_match
153,moncloa_0029,"Notas de 1983 sobre ""Libertad de condenados por el 23-F"", ""Sobre petición de indulto"", ""Vista del recurso de casació...",7_Notas_1983_desp.pdf,None,rtve_1790,"""Notas de 1983 sobre """"Libertad de condenados por el 23-F""""",29_1983_notas_de_1983_sobre_libertad_de_condenados_por_el_23_f_sobre_peticion_de_indulto.pdf,None,0.4299,no_clear_match
154,moncloa_0025,"Juicio del 23-F: acotaciones al desarrollo del juicio, notas procesales sobre el consejo de guerra, nota sobre posib...",3_Juicio_del_23-F_desp.pdf,None,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesales_sobre_el_c.pdf,None,0.3682,no_clear_match


In [72]:
# ------------------------------------------------------------
# 11.12. Documentos RTVE no emparejados tras mejora
# ------------------------------------------------------------

matched_rtve_ids_v2 = set(
    df_source_comparison_v2["matched_rtve_doc_id"]
    .dropna()
    .unique()
)

df_rtve_not_matched_by_moncloa_v2 = df_inventory_rtve[
    ~df_inventory_rtve["doc_id"].isin(matched_rtve_ids_v2)
].copy()

print(f"Documentos RTVE no emparejados tras mejora: {len(df_rtve_not_matched_by_moncloa_v2)}")

df_rtve_not_matched_by_moncloa_v2[
    [
        "doc_id",
        "source_document_id",
        "title",
        "pages",
        "keywords",
        "pdf_url",
    ]
].head(40)

Documentos RTVE no emparejados tras mejora: 12


,doc_id,source_document_id,title,pages,keywords,pdf_url
99,rtve_1761,1761,D.17._AGMAE_R40201_Exp._215,1,S. R. CÂMARA MUNICIPAL DE CAMPO MAIOR SECRETARIA,https://www.rtve.es/contenidos/documentos/23f-desclasificado/154_2026_d17_agmae_r40201_exp_215.pdf
100,rtve_1760,1760,D.16._AGMAE_R40201_Exp._215,1,ASSEMBLEIA MUNICIPAL DO CONCELHO DE CAMPO MAIOR Chanceler do Consulado de Espanha em ELVAS 24. FEB. 1961,https://www.rtve.es/contenidos/documentos/23f-desclasificado/153_2026_d16_agmae_r40201_exp_215.pdf
103,rtve_1757,1757,D.13._AGMAE_R39017_Exp._4,1,Madrid 16 de marzo de 1981 Excmo. Sr. Terence A. Todman,https://www.rtve.es/contenidos/documentos/23f-desclasificado/150_2026_d13_agmae_r39017_exp_4.pdf
105,rtve_1755,1755,D.12._AGMAE_R39017_Exp._4,1,MINISTERIO DE ASUNTOS EXTERIORES Núm. 179 WASHINGTON,https://www.rtve.es/contenidos/documentos/23f-desclasificado/149_2026_d12_agmae_r39017_exp_4.pdf
106,rtve_1754,1754,D.11._AGMAE_R39017_Exp._4,1,JLBB/ng Madrid 10 de marzo de 1931,https://www.rtve.es/contenidos/documentos/23f-desclasificado/148_2026_d11_agmae_r39017_exp_4.pdf
107,rtve_1753,1753,D.10._AGMAE_R39017_Exp._4,1,MINISTERIO DE ASUNTOS EXTERIORES CIFRA Núm. 93 AMERICA DEL NORTE Y PACIFICO,https://www.rtve.es/contenidos/documentos/23f-desclasificado/147_2026_d10_agmae_r39017_exp_4.pdf
108,rtve_1752,1752,D.9._AGMAE_R39017_Exp._4,1,JLBB/mac 63 328(46)-17 Madrid,https://www.rtve.es/contenidos/documentos/23f-desclasificado/146_2026_d9_agmae_r39017_exp_4.pdf
110,rtve_1750,1750,D.7._AGMAE_R39017_Exp._4,1,Mr. Prime Minister President of the Government of Spain Spain,https://www.rtve.es/contenidos/documentos/23f-desclasificado/144_2026_d7_agmae_r39017_exp_4.pdf
111,rtve_1749,1749,D.6._AGMAE_R39017_Exp. 4,1,"Washington, D. C. February 27, 1981 Your Majesty",https://www.rtve.es/contenidos/documentos/23f-desclasificado/143_2026_d6_agmae_r39017_exp_4.pdf
112,rtve_1748,1748,D.5._AGMAE_R39017_Exp._4,1,EMBASSY OF THE UNITED STATES OF AMERICA MADRID THE AMBASSADOR,https://www.rtve.es/contenidos/documentos/23f-desclasificado/142_2026_d5_agmae_r39017_exp_4.pdf


### 11.13. Revisión manual de matches dudosos

Tras mejorar el matching con fechas y asignación uno-a-uno, todos los documentos comparables de La Moncloa quedan asignados a un documento único de RTVE.

Quedan cinco casos con puntuación baja o intermedia. La revisión manual indica que no parecen documentos ausentes, sino coincidencias válidas con menor puntuación debido a:

- títulos largos truncados;
- nombres de archivo poco descriptivos en Moncloa, como `Documento_1_R.pdf`;
- diferencias de redacción entre ambas fuentes.

Por tanto, estos casos se etiquetan como `manual_match`.

La comparación final distingue entre:

- `high_confidence_match`: match automático con alta confianza;
- `manual_match`: match aceptado tras revisión manual;
- `rtve_only`: documento presente en RTVE sin equivalente detectado en Moncloa.

In [73]:
# ------------------------------------------------------------
# 11.13. Revisión manual de matches dudosos
# ------------------------------------------------------------

manual_match_ids = [
    "moncloa_0025",
    "moncloa_0029",
    "moncloa_0030",
    "moncloa_0048",
    "moncloa_0108",
]

df_source_comparison_final = df_source_comparison_v2.copy()

df_source_comparison_final["final_match_status"] = df_source_comparison_final["match_status"]
df_source_comparison_final["manual_review"] = False
df_source_comparison_final["final_notes"] = None

mask_manual = df_source_comparison_final["moncloa_id"].isin(manual_match_ids)

df_source_comparison_final.loc[mask_manual, "final_match_status"] = "manual_match"
df_source_comparison_final.loc[mask_manual, "manual_review"] = True
df_source_comparison_final.loc[
    mask_manual,
    "final_notes"
] = "Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo."

print("Distribución final de estados:")
display(df_source_comparison_final["final_match_status"].value_counts(dropna=False))

df_source_comparison_final[
    df_source_comparison_final["manual_review"]
][
    [
        "moncloa_id",
        "moncloa_title",
        "moncloa_pdf_filename",
        "matched_rtve_doc_id",
        "rtve_title",
        "rtve_pdf_filename",
        "match_score",
        "match_status",
        "final_match_status",
        "final_notes",
    ]
]

Distribución final de estados:


final_match_status
high_confidence_match    150
manual_match               5
Name: count, dtype: int64

,moncloa_id,moncloa_title,moncloa_pdf_filename,matched_rtve_doc_id,rtve_title,rtve_pdf_filename,match_score,match_status,final_match_status,final_notes
150,moncloa_0048,"Estado de opinión sobre las sentencias (sección de contrainformación del EME, sección de contrainteligencia) (7 de j...",Documento_19_R.pdf,rtve_1809,"""Estado de opinión sobre las sentencias (sección de contrainformación del EME",48_1982_estado_de_opinion_sobre_las_sentencias_seccion_de_contrainformacion_del_eme_secc.pdf,0.6094,possible_match,manual_match,Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo.
151,moncloa_0108,"Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle, un cabo 1º, dos guardias y ...",Documento_79_R.pdf,rtve_1710,"""Interesando comparecencia del capitán de la Guardia Civil José Ramón Tostón de la Calle",108_1982_interesando_comparecencia_del_capitan_de_la_guardia_civil_jose_ramon_toston_de_l.pdf,0.5505,possible_match,manual_match,Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo.
152,moncloa_0030,"Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno, el ministro de Defensa y la ...",Documento_1_R.pdf,rtve_1791,"""Guión que sirvió de base para la reunión de S.M. el Rey con el Presidente del Gobierno",30_1981_guion_que_sirvio_de_base_para_la_reunion_de_sm_el_rey_con_el_presidente_del_gobi.pdf,0.5386,possible_match,manual_match,Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo.
153,moncloa_0029,"Notas de 1983 sobre ""Libertad de condenados por el 23-F"", ""Sobre petición de indulto"", ""Vista del recurso de casació...",7_Notas_1983_desp.pdf,rtve_1790,"""Notas de 1983 sobre """"Libertad de condenados por el 23-F""""",29_1983_notas_de_1983_sobre_libertad_de_condenados_por_el_23_f_sobre_peticion_de_indulto.pdf,0.4299,no_clear_match,manual_match,Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo.
154,moncloa_0025,"Juicio del 23-F: acotaciones al desarrollo del juicio, notas procesales sobre el consejo de guerra, nota sobre posib...",3_Juicio_del_23-F_desp.pdf,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesales_sobre_el_c.pdf,0.3682,no_clear_match,manual_match,Match aceptado tras revisión manual por similitud de título/contexto pese a score bajo.


In [74]:
# ------------------------------------------------------------
# 11.14. Guardado de comparación final y documentos RTVE-only
# ------------------------------------------------------------

RTVE_NOT_IN_MONCLOA_PATH = INTERIM_DIR / "rtve_not_matched_by_moncloa.csv"

# Documentos RTVE sin equivalente detectado en Moncloa
df_rtve_not_matched_by_moncloa_final = df_rtve_not_matched_by_moncloa_v2.copy()
df_rtve_not_matched_by_moncloa_final["coverage_status"] = "rtve_only"
df_rtve_not_matched_by_moncloa_final["notes"] = (
    "Documento presente en RTVE sin equivalente detectado en el inventario Moncloa comparable."
)

# Guardar salidas
df_source_comparison_final.to_csv(SOURCE_COMPARISON_PATH, index=False, encoding="utf-8")
df_rtve_not_matched_by_moncloa_final.to_csv(RTVE_NOT_IN_MONCLOA_PATH, index=False, encoding="utf-8")

print(f"Comparación final guardada en: {SOURCE_COMPARISON_PATH.relative_to(PROJECT_ROOT)}")
print(f"RTVE-only guardado en: {RTVE_NOT_IN_MONCLOA_PATH.relative_to(PROJECT_ROOT)}")

print()
print("Resumen final:")
print(f"- Documentos Moncloa comparables: {len(df_source_comparison_final)}")
print(f"- Moncloa cubiertos por RTVE: {df_source_comparison_final['matched_rtve_doc_id'].notna().sum()}")
print(f"- High confidence: {(df_source_comparison_final['final_match_status'] == 'high_confidence_match').sum()}")
print(f"- Manual match: {(df_source_comparison_final['final_match_status'] == 'manual_match').sum()}")
print(f"- RTVE-only: {len(df_rtve_not_matched_by_moncloa_final)}")

Comparación final guardada en: data/interim/source_comparison.csv
RTVE-only guardado en: data/interim/rtve_not_matched_by_moncloa.csv

Resumen final:
- Documentos Moncloa comparables: 155
- Moncloa cubiertos por RTVE: 155
- High confidence: 150
- Manual match: 5
- RTVE-only: 12


### 11.15. Conclusión de la comparación RTVE vs La Moncloa

La comparación entre el inventario oficial de La Moncloa y el corpus construido desde RTVE indica que los documentos comparables de La Moncloa quedan cubiertos por el corpus RTVE.

La Moncloa contiene 156 enlaces PDF detectados, de los cuales uno corresponde al BOE y se excluye de la comparación documental principal. Por tanto, se comparan 155 documentos de La Moncloa contra los 167 documentos obtenidos desde RTVE.

El resultado final de la comparación es:

- 155 documentos comparables de La Moncloa.
- 155 documentos de La Moncloa emparejados con documentos de RTVE.
- 150 coincidencias automáticas de alta confianza.
- 5 coincidencias aceptadas tras revisión manual.
- 12 documentos presentes en RTVE sin equivalente directo detectado en el inventario comparable de La Moncloa.

La diferencia entre RTVE y La Moncloa no indica falta de cobertura en RTVE. Al contrario, para los objetivos del proyecto, RTVE resulta una fuente más completa porque proporciona:

- inventario documental completo;
- metadatos;
- texto completo extraíble;
- enlace final al PDF;
- 12 registros adicionales no detectados en el inventario comparable de La Moncloa.

No se ha realizado una comparación binaria PDF a PDF. La validación se basa en títulos, nombres de archivo, fechas extraídas, similitud textual y revisión manual de los casos dudosos.

Por tanto, no se detecta la necesidad de descargar documentos adicionales desde La Moncloa. La Moncloa queda utilizada como fuente institucional de contraste, mientras que RTVE se mantiene como fuente principal del corpus de trabajo.